# Analysis of American Parlimentary Debate Association rounds Fall 2019 - Spring 2026

## Data Collection and Preprocessing

The following portion of my project deals with data collection and preprocessing following the American Parlimentary Debate Association's (APDA) tournaments, rounds, and debaters. American Parlimentary Debate is a style of debate where one two person team, the Government, proposes a case which is refuted by a second two person team, the Opposition. There are six speeches, and at the conclusion the judge offers a decision on the winner (Gov/Opp), speaks (a score from 15 to 36 for each debater), and ranks (1-4 where each number is assigned to a debater in the round). There are 150+ schools registered with APDA and around six thousand active debaters who have competed in the Spring 2026 semester. The goal of this project is to investigate possible factors affecting novice retention within the league.

The round data for this project was sourced at the APDA online forum where, after the conclusion of each tournament, results are posted in the form of PDF Tab Cards. Tab Cards are structured by team where each team has its own table within the PDF. Each table is labelled with the team name, and contains rows that detail the round number, whether the team was in Government or Opposition position, win/loss status, the opponent team name, judge name, the speaks and ranks for each team member, and the total speaks and ranks of the team. For the preliminary data curation, I used pdfplumber to split each PDF into tables in order to extract data for each team. Bounding boxes were used to extract the team names that prefaced each table. Additionally, a reference was necessary to match the team names, which differ by tournament, to the individual debaters in order to create the columns for the opponent names. It was also necessary to collapse the mirrored rows as each match is represented twice, one for the Government teams table and the other in the Opposition team table. Regex patterns were used for cleaning and normalizing the data. 

Additionally, debater rosters were created by parsing the HTML code of the APDA website's rosters and using Levenshtein distances as well as probability analysis in order to fill in some missing data or correct misspelled names.

## Debate rounds

In [1]:
# imports used for parsing data from PDFs
from pathlib import Path
import re
import logging
import pandas as pd
import pdfplumber

logging.getLogger("pdfminer").setLevel(logging.ERROR)

In [2]:
def get_tab_cards(folder, season, year):
    """
    Creates file paths based on the contents of the data folders. 
    Uses a dictionary to replace file shortenings with the official APDA school name.
    Returns a list for each season's input of the file path, school name, season, and year.
    """
    tab_cards = [] 
    
    for pdf in Path("TabCards/" + folder).glob(f"*_{season}{year}_Tab_Card.pdf"):
        school = pdf.name.replace(f"_{season}{year}_Tab_Card.pdf", "")     
        # The following is a dictionary of names to correct spellings, spacing, and acronyms
        school = {
            "Binghamton": "Binghamton University",
            "BrownAndWesleyan":"Brown + Wesleyan",
            "Chicago":"University of Chicago",
            "ChicagoNortheastern":"University of Chicago + Northeastern",
            "CMU":"Carnegie Mellon",
            "ColumbiaSwarthmore":"Columbia + Swarthmore",
            "CUNYUMD":"CUNY + Maryland",
            "Delaware":"University of Delaware",
            "FranklinAndMarshall": "Franklin and Marshall",
            "Hopkins":"Johns Hopkins",
            "GU":"Georgetown",
            "GW":"George Washington",
            "JHUAU":"Johns Hopkins + American",
            "NU":"Northeastern",
            "NUBC":"Northeastern + Boston College",
            "NYUWashU":"NYU + Washington University",
            "Pitt":"University of Pittsburgh",
            "PittCMU": "University of Pittsburgh + Carnegie Mellon",
            "PrincetonNUBates":"Princeton + Northeastern + Bates",
            "SmithColumbia":"Smith + Columbia",
            "TheCollegeOfNewJersey": "The College of New Jersey",
            "TCNJ": "The College of New Jersey",
            "TempleWesleyan":"Temple + Wesleyan",
            "Tufts2": "Tufts",
            "UMD":"Maryland",
            "UMDBU":"Maryland + Boston University",
            "UMass":"University of Massachusetts",
            "UMass_Amherst":"University of Massachusetts + Amherst",
            "UVA":"University of Virginia",
            "UVAWDS":"University of Virginia + Wellesley",
            "WashU": "Washington University",
            "WestPoint": "West Point",
            "WesleyanTCNJ":"Wesleyan + The College of New Jersey",
            "WilliamAndMary": "William and Mary",
            "William&Mary": "William and Mary",
        }.get(school, school)
        
        tab_cards.append(
            (str(pdf), school, season, year)
        )
    
    return tab_cards

In [3]:
# Initializes tab cards from Fall 2019 - Spring 2026
FALL_2019_TAB_CARDS = get_tab_cards("Fall2019", "Fall", 2019)
SPRING_2020_TAB_CARDS = get_tab_cards("Spring2020", "Spring", 2020)
FALL_2020_TAB_CARDS = get_tab_cards("Fall2020", "Fall", 2020)
SPRING_2021_TAB_CARDS = get_tab_cards("Spring2021", "Spring", 2021)
FALL_2021_TAB_CARDS = get_tab_cards("Fall2021", "Fall", 2021)
SPRING_2022_TAB_CARDS = get_tab_cards("Spring2022", "Spring", 2022)
FALL_2022_TAB_CARDS = get_tab_cards("Fall2022", "Fall", 2022)
SPRING_2023_TAB_CARDS = get_tab_cards("Spring2023", "Spring", 2023)
FALL_2023_TAB_CARDS = get_tab_cards("Fall2023", "Fall", 2023)
SPRING_2024_TAB_CARDS = get_tab_cards("Spring2024", "Spring", 2024)
FALL_2024_TAB_CARDS = get_tab_cards("Fall2024", "Fall", 2024)
SPRING_2025_TAB_CARDS = get_tab_cards("Spring2025", "Spring", 2025)
FALL_2025_TAB_CARDS = get_tab_cards("Fall2025", "Fall", 2025)
SPRING_2026_TAB_CARDS = get_tab_cards("Spring2026", "Spring", 2026)

Several tournaments were excluded for bad formating, lack of tab cards, or lack of permission to the documnets. The following describes those ommited:

In [4]:
# Formatting of the excluded tournaments
from tabulate import tabulate
omitted_tables = [["Fall2019","Fordham, UVA, Columbia","",""],["Spring2020","","GU, Williams",""],
                  ["Fall2020","","Harvard",""],["Spring2021","Swarthmore","Brandeis, W&M, GU, Yale",""],
                  ["Fall2021","","","Tufts/UMD, GW/Fordham, Brown, Harvard/Penn"],["Spring2022","Princeton, Yale, Rutgers","Brown, Williams","Darthmouth"],
                  ["Fall2022","","Brown, Yale, Williams","Tufts, Harvard, Rutgers"],["Spring2023","Hopkins","Brandeis, Rutgers, Penn, Williams, UChicago","UMass"],
                  ["Fall2023","Hopkins, Swarthmore","Brandeis, Harvard",""],["Spring2024","Amherst","UMass, Brandeis, NYU, Windsor, Tufts",""],
                  ["Fall2024","","Harvard, Binghamton, Tufts, Brown",""],["Spring2025","Rutgers, Amherst, Penn, UVA","","Dartmouth, UT Austin"],
                  ["Fall2025","Drexel","","Brandeis, Bates, American"],["Spring2026","Temple","",""]]
print(tabulate(omitted_tables, headers=["season","Bad Format", "Lacking Permission", "Missing Tab Card"], tablefmt="grid"))

+------------+-----------------------------+---------------------------------------------+--------------------------------------------+
| Season     | Bad Format                  | Lacking Permission                          | Missing Tab Card                           |
+============+=============================+=============================================+============================================+
| Fall2019   | Fordham, UVA, Columbia      |                                             |                                            |
+------------+-----------------------------+---------------------------------------------+--------------------------------------------+
| Spring2020 |                             | GU, Williams                                |                                            |
+------------+-----------------------------+---------------------------------------------+--------------------------------------------+
| Fall2020   |                             | Har

In order to clean and sanitize the data, which is entered by college students and often has errors as well as embelishment (emojis, nicknames), I created a few regex patterns used throughout my parsing and normalization functions.

In [5]:
# pre-compile regex patterns
MULTISPACE_RX = re.compile(r"\s+")
STATUS_PAREN_RX = re.compile(r"\s*\(([NV])\)\s*$")
CLEAN_TRAILING_RX = re.compile(r"\s+$")
TEAM_PREFIX_RX = re.compile(r"Team:\s*(.+)")
EMOJI_RX = re.compile(
    r"[\U00010000-\U0010FFFF"  
    r"\u2300-\u23FF"           
    r"\u2600-\u26FF"           
    r"\u2700-\u27BF"          
    r"\u2B00-\u2BFF"           
    r"\uFE0F"                  
    r"\u200D]"                 
)

NUMBER_RX = re.compile(r"\d")
KEYCAP_RX = re.compile(r"[\u20E3]")
TM_RX = re.compile(r"™")
QUOTES_RX = re.compile(r'"[^"]*"')

The next section outlines the functions that comprise my PDF scraping, parsing, and cleaning. With my parser, I find the bounding boxes of each tables in order to find the data for each unique team, even across page breaks, and identify the team names for use of cross reference later. The parser additionally normalizes names and number columns utilizing the regex patterns. In order to clean the data to be more managable to analyze, it also splits individuals' names from their statuses (varsity or novice), as well as speaks from ranks. These values are stored in four separate columns. Finally, a registry of the team names as well as the members of the team are stored in order to cross reference and identify the opponents names rather than the team name, which is non-unique and pairings change per tournament.

In [12]:
def normalize_cell(name):
    """
    Fills NaN with an empty string, removes emojis, and removes spaces
    """
    if name is None:
        return ""

    name = str(name).title()
    name = EMOJI_RX.sub("", name)
    name = KEYCAP_RX.sub("", name)
    name = TM_RX.sub("", name)

    return MULTISPACE_RX.sub(" ", name).strip()

def normalize_names(name):
    """
    Removes numbers from name entries
    """
    if name is None:
        return ""

    name = str(name)
    name = NUMBER_RX.sub("", name)
    name = QUOTES_RX.sub("", name)

    return normalize_cell(name)
    
def split_name_and_status(raw_name):
    """
    Return (clean_name, status) extracted from a speaker header.
    """
    if not raw_name:
        return "", None

    name = normalize_names(raw_name)
    match = STATUS_PAREN_RX.search(name)

    if match:
        status = "Novice" if match.group(1) == "N" else "Varsity"
        return STATUS_PAREN_RX.sub("", name).strip(), status

    return name, None

def split_speaks_and_ranks(df, score_col, prefix):
    """
    Split a score column into separate speaks and rank columns.
    """
    if score_col not in df.columns:
        return df

    cleaned_series = df[score_col].astype(str).str.replace(r"[\(\)\s]", "", regex=True)
    split_data = cleaned_series.str.split(",", expand=True)

    if split_data.shape[1] < 2:
        split_data = pd.DataFrame(index=df.index, columns=[0, 1])

    df[f"{prefix}_speaks"] = pd.to_numeric(split_data[0], errors="coerce")
    df[f"{prefix}_rank"] = pd.to_numeric(split_data[1], errors="coerce").astype("Int64") # Capital I allows NaN integers

    return df.drop(columns=[score_col])

def add_speaker_and_opponent_names(all_df, speaker_df):
    """
    Matches teams and opponents using normalized team keys, then splits
    speaker scores into speaks and rank columns.
    """
    speaker_df = speaker_df.copy()
    speaker_df["_key"] = speaker_df["team"].map(normalize_names)
    speaker_df = speaker_df.drop_duplicates(subset="_key", keep="first")

    lookup = speaker_df.set_index("_key")[
        ["speaker_one_name", "speaker_one_status", "speaker_two_name", "speaker_two_status"]
    ]

    opponent_lookup = lookup.rename(columns={
        "speaker_one_name": "opponent_one_name",
        "speaker_one_status": "opponent_one_status",
        "speaker_two_name": "opponent_two_name",
        "speaker_two_status": "opponent_two_status"
    })
    
    df = all_df.copy()
    df["judge"] = df["judge"].map(normalize_names)
    df["_team_key"] = df["team"].map(normalize_names)
    df["_opponent_key"] = df["opponent"].map(normalize_names)
  
    df = df.merge(lookup, left_on="_team_key", right_index=True, how="left")
    df = df.merge(opponent_lookup, left_on="_opponent_key", right_index=True, how="left")


    score_lookup = all_df.copy()
    score_lookup["_opponent_key"] = score_lookup["team"].map(normalize_names)
    
    score_lookup = (
        score_lookup[
            ["_opponent_key", "round", "speaker_one_score", "speaker_two_score"]
        ]
        .drop_duplicates(["_opponent_key", "round"])
        .rename(columns={
            "speaker_one_score": "opponent_speaker_one_score",
            "speaker_two_score": "opponent_speaker_two_score"
        })
    )
    df = df.merge(score_lookup, on=["_opponent_key", "round"], how="left")
    df = df.drop(columns=["team", "opponent", "_team_key", "_opponent_key"])

    df = split_speaks_and_ranks(df, "speaker_one_score", "speaker_one")
    df = split_speaks_and_ranks(df, "speaker_two_score", "speaker_two")
    df = split_speaks_and_ranks(df, "opponent_speaker_one_score", "opponent_one")
    df = split_speaks_and_ranks(df, "opponent_speaker_two_score", "opponent_two")
    
    ordered_cols = [c for c in [
        "tournament", "season", "year", "round", "g_o", "w_l",
        "speaker_one_name", "speaker_one_status", "speaker_two_name", "speaker_two_status",
        "opponent_one_name", "opponent_one_status", "opponent_two_name", "opponent_two_status",
        "judge", "speaker_one_speaks", "speaker_one_rank", "speaker_two_speaks", "speaker_two_rank",
        "opponent_one_speaks", "opponent_one_rank", "opponent_two_speaks", "opponent_two_rank", "total",
    ] if c in df.columns]
    
    return df[ordered_cols]

def parse_tab_cards(card):
    """
    Parses the individual tournament tab card
    """
    all_rows, speaker_lookup = [], []
    seen_teams = set()
    current_team, current_speakers, pending_label = None, None, None
    
    with pdfplumber.open(card) as pdf:
        for page in pdf.pages:
            team_labels = [] # dictionaries for team names + locations
            for line in page.extract_text_lines():
                match = TEAM_PREFIX_RX.search(line["text"])
                if match:
                    team_labels.append({"name": match.group(1).strip(), "top": line["top"]})
 
            tables = page.find_tables()
 
            for table in tables:
                data = table.extract()
                if not data or len(data) < 1:
                    continue
    
                first_cell = normalize_cell(data[0][0]) if data[0] else ""
                has_header = (first_cell in ("R", "")) # stores whether the first cell is the beginning of the table 
 
                if has_header: # if a new table, store 
                    header = [normalize_cell(c) for c in data[0]]
                    body = data[1:]
                    speaker1_header = header[5] if len(header) > 5 else ""
                    speaker2_header = header[6] if len(header) > 6 else ""
                else: # if a continuation of the last table
                    body = data
                    speaker1_header, speaker2_header = current_speakers if current_speakers else ("", "")

                # finds the team name stored in team_labels that is above current table
                table_top = table.bbox[1]
                labels_above = [t for t in team_labels if t["top"] <= table_top]
 
                if labels_above:
                    team_name = normalize_cell(
                        max(labels_above, key=lambda t: t["top"])["name"]
                    )
                    current_team = team_name
                    current_speakers = (speaker1_header, speaker2_header)
                    pending_label = None
                elif has_header and current_speakers and (speaker1_header, speaker2_header) == current_speakers: # handles table breaking to next page with label headings
                    team_name = current_team
                elif not has_header and current_team is not None: # handles continuation table w/o header
                    team_name = current_team
                elif pending_label is not None: # detected team name before
                    team_name = pending_label
                    current_team = team_name
                    current_speakers = (speaker1_header, speaker2_header)
                    pending_label = None
                else:
                    team_name = current_team
                    current_speakers = (speaker1_header, speaker2_header)

                if team_name not in seen_teams:
                    seen_teams.add(team_name) # set for O(1) lookup
                    name1, status1 = split_name_and_status(speaker1_header)
                    name2, status2 = split_name_and_status(speaker2_header)
                    speaker_lookup.append({
                        "team": team_name,
                        "speaker_one_name": name1, "speaker_one_status": status1,
                        "speaker_two_name": name2, "speaker_two_status": status2,
                    })
 
                for row in body:
                    row = [normalize_cell(c) for c in row]
                    if not row or row[0].lower().startswith("tournament totals"):
                        continue  
 
                    row = (row + [""] * 8)[:8]
                    round_no, g_o, w_l, opponent, judge, sp1, sp2, total = row
 
                    if not round_no:
                        if not any([g_o, w_l, opponent, judge, sp1, sp2]):
                            continue
                        round_no = "UNKNOWN (split across page break)"
 
                    all_rows.append({
                        "team": team_name, "round": round_no, "g_o": g_o, "w_l": w_l,
                        "opponent": opponent, "judge": judge, 
                        "speaker_one_score": sp1, "speaker_two_score": sp2, "total": total,
                    })
 
            if team_labels:
                last_label = max(team_labels, key=lambda t: t["top"])
                table_tops = [t.bbox[1] for t in tables]
                if not any(last_label["top"] <= top for top in table_tops):
                    pending_label = normalize_cell(last_label["name"])
 
    return pd.DataFrame(all_rows), pd.DataFrame(speaker_lookup)

def process_all_cards(tab_card):
    """
    Calls parse then adds Tournament, season, and year column while dropping Total. Combines all semester rows + speakers into one row
    """
    semester_rounds_df = [] 
    semester_speakers_df = [] 
 
    for pdf_path, tournament_name, season, year in tab_card:
        tournament_rounds_df, tournament_speaker_df = parse_tab_cards(pdf_path)
        
        if tournament_rounds_df.empty:
            continue
        tournament_rounds_df = tournament_rounds_df.drop(columns=["total"], errors="ignore")
        tournament_rounds_df.insert(0, "tournament", tournament_name)
        tournament_rounds_df.insert(1, "season", season)
        tournament_rounds_df.insert(2, "year", year)

        tournament_rounds_df = add_speaker_and_opponent_names(tournament_rounds_df, tournament_speaker_df)
        semester_rounds_df.append(tournament_rounds_df)
 
        tournament_speaker_df.insert(0, "tournament", tournament_name)
        tournament_speaker_df.insert(1, "season", season)
        tournament_speaker_df.insert(2, "year", year)
        
        semester_speakers_df.append(tournament_speaker_df)
 
    return pd.concat(semester_rounds_df, ignore_index=True), pd.concat(semester_speakers_df, ignore_index=True)

The following code utilizes the PDF scraping, parsing, and cleaning functions as well as the tab card paths in order to populate the dataframe.

In [102]:
# Assigns a label for each semester of tab cards
semester_tab_paths =  [(FALL_2019_TAB_CARDS, "Fall_2019.csv"), 
                       (SPRING_2020_TAB_CARDS, "Spring_2020.csv"), (FALL_2020_TAB_CARDS, "Fall_2020.csv"),
                       (SPRING_2021_TAB_CARDS, "Spring_2021.csv"), (FALL_2021_TAB_CARDS, "Fall_2021.csv"),
                       (SPRING_2022_TAB_CARDS, "Spring_2022.csv"), (FALL_2022_TAB_CARDS, "Fall_2022.csv"),
                       (SPRING_2023_TAB_CARDS, "Spring_2023.csv"), (FALL_2023_TAB_CARDS, "Fall_2023.csv"),
                       (SPRING_2024_TAB_CARDS, "Spring_2024.csv"), (FALL_2024_TAB_CARDS, "Fall_2024.csv"),
                       (SPRING_2025_TAB_CARDS, "Spring_2025.csv"), (FALL_2025_TAB_CARDS, "Fall_2025.csv"),
                       (SPRING_2026_TAB_CARDS, "Spring_2026.csv")
                      ]
# Creates a df of the rounds and a speaker_df with all speakers seen
dfs = []
speakers = []
for (semester_card, label) in semester_tab_paths:
    df, speaker_df = process_all_cards(semester_card)

    # Cleans df by removing BYE rows or rows where speaker names are missing or ranks/speaks are zero
    df = df[
        ~df["g_o"].fillna("").str.strip().isin(["", "BYE"])
        & df["speaker_one_name"].fillna("").str.strip().ne("")
        & df["speaker_two_name"].fillna("").str.strip().ne("")
        & df["opponent_one_name"].fillna("").str.strip().ne("")
        & df["opponent_two_name"].fillna("").str.strip().ne("")
        & (df["speaker_one_rank"] != 0)
        & (df["speaker_two_rank"] != 0)
        & (df["opponent_one_rank"] != 0)
        & (df["opponent_two_rank"] != 0)
        & (df["speaker_one_speaks"] != 0.0)
        & (df["speaker_two_speaks"] != 0.0)
        & (df["opponent_one_speaks"] != 0.0)
        & (df["opponent_two_speaks"] != 0.0)
        & (df["w_l"] != "Wf")
        & (df["w_l"] != "Lf")
    ]
    df["judge"] = df["judge"].map(normalize_names)
    df['judge'] = (
        df['judge']
        .str.split(' - ').str[0]
        .str.replace(r'\s*\(V\)\s*', '', regex=True)
        .str.replace(r"\s*-\s*", "-", regex=True)
        .str.strip()
    )
    df["speaker_one_name"] = df["speaker_one_name"].str.replace(r"\s*-\s*", "-", regex=True)
    df["speaker_two_name"] = df["speaker_two_name"].str.replace(r"\s*-\s*", "-", regex=True)
    df["opponent_one_name"] = df["opponent_one_name"].str.replace(r"\s*-\s*", "-", regex=True)
    df["opponent_two_name"] = df["opponent_two_name"].str.replace(r"\s*-\s*", "-", regex=True)
    

    #df.to_csv(label, index=False)
    dfs.append(df)
    speakers.append(speaker_df)

Due to the structure of the tab cards, each unique round is seen by two perspectives, once under the Government team's tab card and once under the Opposition team's tab card. In order to analyze the data, I first want to collapse the dataframe to see each round only once. To accomplish this, the following function takes the two "mirror" rounds and saves only the winning perspective. If no mirror round is found, it preserves the data.

In [108]:
def collapse_mirrored_rounds(df):
    """
    Collapses mirrored round-pairs (same round logged once per team's
    perspective) into a single row per round, keeping the winning team's
    perspective. Falls back to keeping the row as-is if no clear winner
    is found (e.g. missing/unexpected W/L codes) or if the round was
    only logged once to begin with.
    """
    df = df.copy()
    df["w_l"] = df["w_l"].str.replace("Aw", "W")

    def team_key(a, b):
        return tuple(sorted([str(a).strip().lower(), str(b).strip().lower()]))

    df["_team1"] = df.apply(lambda r: team_key(r["speaker_one_name"], r["speaker_two_name"]), axis=1)
    df["_team2"] = df.apply(lambda r: team_key(r["opponent_one_name"], r["opponent_two_name"]), axis=1)
    df["_match_key"] = df.apply(lambda r: tuple(sorted([r["_team1"], r["_team2"]])), axis=1)
    df["_context_key"] = list(zip(df["tournament"], df["season"], df["year"], df["round"]))

    kept_rows = []

    for _, group in df.groupby(["_context_key", "_match_key"], sort=False):
        if len(group) == 1:
            kept_rows.append(group.iloc[0])
            continue

        winners = group[group["w_l"] == "W"]

        if len(winners) == 1:
            kept_rows.append(winners.iloc[0])
        else:
            kept_rows.append(group.iloc[0])

    result = pd.DataFrame(kept_rows).drop(columns=["_team1", "_team2", "_match_key", "_context_key"])
    return result.reset_index(drop=True)

In [109]:
rounds_dfs = pd.concat(dfs, ignore_index=True)
collapsed_rounds_dfs = collapse_mirrored_rounds(rounds_dfs)
collapsed_rounds_dfs.to_csv("Data_Curation_Data/Rounds.csv", index=False)

In [110]:
print(collapsed_rounds_dfs.isna().sum())
print(collapsed_rounds_dfs[collapsed_rounds_dfs.isna().any(axis=1)])

tournament              0
season                  0
year                    0
round                   0
g_o                     0
w_l                     0
speaker_one_name        0
speaker_one_status     13
speaker_two_name        0
speaker_two_status     13
opponent_one_name       0
opponent_one_status     7
opponent_two_name       0
opponent_two_status     7
judge                   0
speaker_one_speaks      0
speaker_one_rank        0
speaker_two_speaks      0
speaker_two_rank        0
opponent_one_speaks     0
opponent_one_rank       0
opponent_two_speaks     0
opponent_two_rank       0
dtype: int64
      tournament season  year round g_o w_l          speaker_one_name  \
13523      Brown   Fall  2025     1   O   W            Alan Cai (N) (   
13524      Brown   Fall  2025     2   O   W  Khumoetsile Molefakgotla   
13525      Brown   Fall  2025     3   G   W            Leo Nitczynski   
13526      Brown   Fall  2025     4   G   L            Alan Cai (N) (   
13527      Brown   Fall 

From the above, we can see that there were a few parsing errors. But out of thousands of rows, only a few cells were lost. I will go in manually to correct these.

In [111]:
collapsed_rounds_dfs["speaker_one_name"] = collapsed_rounds_dfs["speaker_one_name"].str.replace("Alan Cai (N) (", "Alan Cai", regex=False)
collapsed_rounds_dfs["speaker_two_name"] = collapsed_rounds_dfs["speaker_two_name"].str.replace("Ozodbek Eshboboev N)", "Ozodbek Eshboboev", regex=False)
collapsed_rounds_dfs["opponent_one_name"] = collapsed_rounds_dfs["opponent_one_name"].str.replace("Alan Cai (N) (", "Alan Cai", regex=False)
collapsed_rounds_dfs["opponent_two_name"] = collapsed_rounds_dfs["opponent_two_name"].str.replace("Ozodbek Eshboboev N)", "Ozodbek Eshboboev", regex=False)
collapsed_rounds_dfs["speaker_one_name"] = collapsed_rounds_dfs["speaker_one_name"].str.replace("Emma Listgarten (V) (", "Emma Listgarten", regex=False)
collapsed_rounds_dfs["speaker_two_name"] = collapsed_rounds_dfs["speaker_two_name"].str.replace("Tanya Chatterjee V)", "Tanya Chatterjee", regex=False)
collapsed_rounds_dfs["speaker_one_name"] = collapsed_rounds_dfs["speaker_one_name"].str.replace("Maneet Mehta (N) (", "Maneet Mehta", regex=False)
collapsed_rounds_dfs["speaker_two_name"] = collapsed_rounds_dfs["speaker_two_name"].str.replace("Midhun Sadanand N)", "Midhun Sadanand", regex=False)
collapsed_rounds_dfs["opponent_one_name"] = collapsed_rounds_dfs["opponent_one_name"].str.replace("Maneet Mehta (N) (", "Maneet Mehta", regex=False)
collapsed_rounds_dfs["opponent_two_name"] = collapsed_rounds_dfs["opponent_two_name"].str.replace("Midhun Sadanand N)", "Midhun Sadanand", regex=False)
collapsed_rounds_dfs["speaker_one_name"] = collapsed_rounds_dfs["speaker_one_name"].str.replace("Flo Arnado (N) (", "Flo Arnado", regex=False)
collapsed_rounds_dfs["speaker_two_name"] = collapsed_rounds_dfs["speaker_two_name"].str.replace("Jiya Mahapatra N)", "Jiya Mahapatra", regex=False)
collapsed_rounds_dfs["opponent_one_name"] = collapsed_rounds_dfs["opponent_one_name"].str.replace("Flo Arnado (N) (", "Flo Arnado", regex=False)
collapsed_rounds_dfs["opponent_two_name"] = collapsed_rounds_dfs["opponent_two_name"].str.replace("Jiya Mahapatra N)", "Jiya Mahapatra", regex=False)

collapsed_rounds_dfs.loc[(collapsed_rounds_dfs['tournament'] == 'Brown') & (collapsed_rounds_dfs['season'] == 'Fall') & (collapsed_rounds_dfs['year'] == 2025) &  ((collapsed_rounds_dfs['round'] == '5') | (collapsed_rounds_dfs['round'] == '4') | (collapsed_rounds_dfs['round'] == '1')) &  (collapsed_rounds_dfs['speaker_one_name'] == 'Alan Cai'), 'speaker_one_status'] = 'Novice'
collapsed_rounds_dfs.loc[(collapsed_rounds_dfs['tournament'] == 'Brown') & (collapsed_rounds_dfs['season'] == 'Fall') & (collapsed_rounds_dfs['year'] == 2025) &  ((collapsed_rounds_dfs['round'] == '5') | (collapsed_rounds_dfs['round'] == '4') | (collapsed_rounds_dfs['round'] == '1')) & (collapsed_rounds_dfs['speaker_two_name'] == 'Ozodbek Eshboboev'), 'speaker_two_status'] = 'Novice'
collapsed_rounds_dfs.loc[(collapsed_rounds_dfs['tournament'] == 'Brown') & (collapsed_rounds_dfs['season'] == 'Fall') & (collapsed_rounds_dfs['year'] == 2025) &  ((collapsed_rounds_dfs['round'] == '2') | (collapsed_rounds_dfs['round'] == '3')) &  (collapsed_rounds_dfs['opponent_one_name'] == 'Alan Cai'), 'opponent_one_status'] = 'Novice'
collapsed_rounds_dfs.loc[(collapsed_rounds_dfs['tournament'] == 'Brown') & (collapsed_rounds_dfs['season'] == 'Fall') & (collapsed_rounds_dfs['year'] == 2025) &  ((collapsed_rounds_dfs['round'] == '2') | (collapsed_rounds_dfs['round'] == '3')) & (collapsed_rounds_dfs['opponent_two_name'] == 'Ozodbek Eshboboev'), 'opponent_two_status'] = 'Novice'
collapsed_rounds_dfs.loc[(collapsed_rounds_dfs['tournament'] == 'Brown') & (collapsed_rounds_dfs['season'] == 'Fall') & (collapsed_rounds_dfs['year'] == 2025) &  ((collapsed_rounds_dfs['round'] == '1') | (collapsed_rounds_dfs['round'] == '2') | (collapsed_rounds_dfs['round'] == '3') | (collapsed_rounds_dfs['round'] == '4') | (collapsed_rounds_dfs['round'] == '5')) &  (collapsed_rounds_dfs['speaker_one_name'] == 'Emma Listgarten'), 'speaker_one_status'] = 'Varsity'
collapsed_rounds_dfs.loc[(collapsed_rounds_dfs['tournament'] == 'Brown') & (collapsed_rounds_dfs['season'] == 'Fall') & (collapsed_rounds_dfs['year'] == 2025) &  ((collapsed_rounds_dfs['round'] == '1') | (collapsed_rounds_dfs['round'] == '2') | (collapsed_rounds_dfs['round'] == '3') | (collapsed_rounds_dfs['round'] == '4') | (collapsed_rounds_dfs['round'] == '5')) &  (collapsed_rounds_dfs['speaker_two_name'] == 'Tanya Chatterjee'), 'speaker_two_status'] = 'Varsity'
collapsed_rounds_dfs.loc[(collapsed_rounds_dfs['tournament'] == 'Brown') & (collapsed_rounds_dfs['season'] == 'Fall') & (collapsed_rounds_dfs['year'] == 2025) &  ((collapsed_rounds_dfs['round'] == '5') | (collapsed_rounds_dfs['round'] == '3') | (collapsed_rounds_dfs['round'] == '1')) &  (collapsed_rounds_dfs['speaker_one_name'] == 'Maneet Mehta'), 'speaker_one_status'] = 'Novice'
collapsed_rounds_dfs.loc[(collapsed_rounds_dfs['tournament'] == 'Brown') & (collapsed_rounds_dfs['season'] == 'Fall') & (collapsed_rounds_dfs['year'] == 2025) &  ((collapsed_rounds_dfs['round'] == '5') | (collapsed_rounds_dfs['round'] == '3') | (collapsed_rounds_dfs['round'] == '1')) & (collapsed_rounds_dfs['speaker_two_name'] == 'Midhun Sadanand'), 'speaker_two_status'] = 'Novice'
collapsed_rounds_dfs.loc[(collapsed_rounds_dfs['tournament'] == 'Brown') & (collapsed_rounds_dfs['season'] == 'Fall') & (collapsed_rounds_dfs['year'] == 2025) &  ((collapsed_rounds_dfs['round'] == '2') | (collapsed_rounds_dfs['round'] == '4')) &  (collapsed_rounds_dfs['opponent_one_name'] == 'Maneet Mehta'), 'opponent_one_status'] = 'Novice'
collapsed_rounds_dfs.loc[(collapsed_rounds_dfs['tournament'] == 'Brown') & (collapsed_rounds_dfs['season'] == 'Fall') & (collapsed_rounds_dfs['year'] == 2025) &  ((collapsed_rounds_dfs['round'] == '2') | (collapsed_rounds_dfs['round'] == '4')) & (collapsed_rounds_dfs['opponent_two_name'] == 'Midhun Sadanand'), 'opponent_two_status'] = 'Novice'
collapsed_rounds_dfs.loc[(collapsed_rounds_dfs['tournament'] == 'Brown') & (collapsed_rounds_dfs['season'] == 'Fall') & (collapsed_rounds_dfs['year'] == 2025) &  ((collapsed_rounds_dfs['round'] == '1') | (collapsed_rounds_dfs['round'] == '4')) &  (collapsed_rounds_dfs['speaker_one_name'] == 'Flo Arnado'), 'speaker_one_status'] = 'Novice'
collapsed_rounds_dfs.loc[(collapsed_rounds_dfs['tournament'] == 'Brown') & (collapsed_rounds_dfs['season'] == 'Fall') & (collapsed_rounds_dfs['year'] == 2025) &  ((collapsed_rounds_dfs['round'] == '1') | (collapsed_rounds_dfs['round'] == '4')) & (collapsed_rounds_dfs['speaker_two_name'] == 'Jiya Mahapatra'), 'speaker_two_status'] = 'Novice'
collapsed_rounds_dfs.loc[(collapsed_rounds_dfs['tournament'] == 'Brown') & (collapsed_rounds_dfs['season'] == 'Fall') & (collapsed_rounds_dfs['year'] == 2025) &  ((collapsed_rounds_dfs['round'] == '2') | (collapsed_rounds_dfs['round'] == '3') | (collapsed_rounds_dfs['round'] == '5')) &  (collapsed_rounds_dfs['opponent_one_name'] == 'Flo Arnado'), 'opponent_one_status'] = 'Novice'
collapsed_rounds_dfs.loc[(collapsed_rounds_dfs['tournament'] == 'Brown') & (collapsed_rounds_dfs['season'] == 'Fall') & (collapsed_rounds_dfs['year'] == 2025) &  ((collapsed_rounds_dfs['round'] == '2') | (collapsed_rounds_dfs['round'] == '3') | (collapsed_rounds_dfs['round'] == '5')) & (collapsed_rounds_dfs['opponent_two_name'] == 'Jiya Mahapatra'), 'opponent_two_status'] = 'Novice'


In [112]:
print(collapsed_rounds_dfs.isna().sum())
#collapsed_rounds_dfs = collapsed_rounds_dfs[collapsed_rounds_dfs.isna().any(axis=1)]
collapsed_rounds_dfs.to_csv("Data_Curation_Data/Rounds.csv", index=False)

tournament             0
season                 0
year                   0
round                  0
g_o                    0
w_l                    0
speaker_one_name       0
speaker_one_status     0
speaker_two_name       0
speaker_two_status     0
opponent_one_name      0
opponent_one_status    0
opponent_two_name      0
opponent_two_status    0
judge                  0
speaker_one_speaks     0
speaker_one_rank       0
speaker_two_speaks     0
speaker_two_rank       0
opponent_one_speaks    0
opponent_one_rank      0
opponent_two_speaks    0
opponent_two_rank      0
dtype: int64


So all the errors were from the same Brown Fall 2025 tournament and have now been corrected! Now we can move on to the roster section.

## Roster of Debaters

The next table we need to construct is a list of debaters and the schools they are from. To accomplish this I will first extract the debate rosters directly from the APDA website, then I will utiilize the Levenshtein distance algorithm in order to identify debaters and judges that may have been misspelled or use nicknames and replace these variations with the names registered with APDA. Finally, I will compare the schools of previous partners of each individual competing that was not claimed by any official school roster in order to construct a possible list debaters with school associations.

In [113]:
from pathlib import Path
from bs4 import BeautifulSoup
from io import StringIO
import pandas as pd

First, we need to create the paths to the HTML pages, much like with the tab cards in order to parse the pages to get our base roster.

In [114]:
def get_debaters(school, year):
    """
    Creates file paths based on the contents of the data folder. 
    Uses a dictionary to replace file shortenings with the official APDA school name.
    Returns the file path.
    """
    paths = []
    for html in Path("Debaters").glob(f"{school}_{year}.html"):
        school = html.name.replace(f"_{year}.html", "")     
        school = {
            "BentleyUniversity": "Bentley University",
            "BGC": "Bard Graduate Center",
            "BinghamtonUniversity": "Binghamton University",
            "BostonCollege": "Boston College",
            "BostonUniversity": "Boston University",
            "BowdoinCollege": "Bowdoin College",
            "BPP": "Brierley Price Prior",
            "BrynMawr": "Bryn Mawr",
            "CarnegieMellon": "Carnegie Mellon",
            "CCSF": "City College of San Francisco",
            "ColumbiaLaw": "Columbia Law",
            "DuquesneUniversity": "Duquesne University",
            "FisherCollege":"Fisher College",
            "FIU": "Florida International University",
            "FloridaStateUniversity": "Florida State University",
            "FranklinandMarshall": "Franklin and Marshall",
            "GeorgeMason": "George Mason",
            "GeorgeWashington": "George Washington",
            "GroveCityCollege": "Grove City College",
            "HartHouse": "Hart House",
            "HarvardLaw": "Harvard Law",
            "HWS": "Hobart and William Smith",
            "HobartandWilliamSmith": "Hobart and William Smith",
            "IBADU": "University of Dhaka",
            "JohnsHopkins": "Johns Hopkins",
            "KwameNkrumahUniversityofScienceandTechnology": "Kwame Nkrumah University of Science and Technology",
            "LaVerne": "La Verne",
            "LoyolaMarymount": "Loyola Marymount",
            "LoyolaUniversityChicago": "Loyola University Chicago",
            "MoodyBibleInstitute": "Moody Bible Institute",
            "MorehouseCollege": "Morehouse College",
            "MountHolyoke": "Mount Holyoke",
            "NotreDame": "Notre Dame",
            "PatrickHenry": "Patrick Henry",
            "PrinceGeorgesCommunityCollege": "Prince George's Community College",
            "ProvidenceCollege": "Providence College",
            "QueensUniversity": "Queen's University",
            "SanJoseState": "San Jose State",
            "SantaClara":"Santa Clara",
            "SimonFraserUniversity": "Simon Fraser University",
            "SimonsRockCollege": "Simon's Rock College",
            "SouthCarolina": "South Carolina",
            "Sydney": "University of Sydney",
            "StAndrews": "St. Andrews",
            "StJohns": "St. Johns",
            "StMarys": "St. Mary's",
            "StonyBrookUniversity": "Stony Brook University",
            "TelAviv": "Tal Aviv",
            "TheCollegeofNewJersey": "The College of New Jersey",
            "UCDL&H": "UCD L&H",
            "UniversityofAlaskaAnchorage": "University of Alaska Anchorage",
            "UniversityofAlbany": "University of Albany",
            "UniversityofBritishColumbia": "University of British Columbia",
            "UniversityofCalgary": "University of Calgary",
            "UniversityofChicago": "University of Chicago",
            "UniversityofDelaware": "University of Delaware",
            "UniversityofDenver": "University of Denver",
            "UniversityofGuelph": "University of Guelph",
            "UniversityofHawaiiatManoa": "University of Hawaii at Manoa",
            "UniversityofMassachusetts": "University of Massachusetts",
            "UniversityofMichigan": "University of Michigan",
            "UniversityofMinnesota": "University of Minnesota",
            "UniversityofNewSouthWales": "University of New South Wales",
            "UniversityofNorthCarolina": "University of North Carolina",
            "UniversityofPittsburgh": "University of Pittsburgh",
            "UniversityofSouthernCalifornia": "University of Southern California",
            "UniversityofSydney": "University of Sydney",
            "UniversityofthePeople": "University of the People",
            "UniversityofVermont": "University of Vermont",
            "UniversityofVirginia": "University of Virginia",
            "UniversityofWaterloo": "University of Waterloo",
            "UTAustin": "UT Austin",
            "WashingtonUniversityinStLouis": "Washington University in St. Louis",
            "WilfredLaurierUniversity": "Wilfred Laurier University",
            "WilliamandMary": "William and Mary",
            "YorkUniversity": "York University",
        }.get(school, school)
        
        paths.append((html, school, year))
    return paths

The following extracts and stores the rosters for each school.

In [115]:
# Assigns a label for each school's roster
schools = [("American", "AMERICAN"), ("Adelphi", "ADELPHI"), ("Amherst", "AMHERST"), ("Bard", "BARD"), 
           ("Bates", "Bates"), ("BentleyUniversity", "BENTLEYUNIVERSITY"), ("Berkeley", "BERKELEY"), 
           ("BGC", "BGC"), ("BinghamtonUniversity", "BINGHAMTONUNIVERSITY"), ("BostonCollege", "BOSTONCOLLEGE"), 
           ("BostonUniversity", "BOSTONUNIVERSITY"), ("BowdoinCollege", "BOWDOINCOLLEGE"), ("BPP", "BPP"), 
           ("Bradley", "BRADLEY"), ("Brandeis", "BRANDEIS"), ("Brown", "BROWN"), ("BrynMawr", "BRYNMAWR"), ("Bucknell", "BUCKNELL"),
           ("Cambridge", "CAMBRIDGE"), ("Carleton", "CARLETON"), ("CarnegieMellon", "Carnegie Mellon"), ("CCSF", "CCSF"),
           ("Claremont", "CLAREMONT"), ("Colgate", "COLGATE"), ("Columbia", "COLUMBIA"), ("ColumbiaLaw", "COLUMBIALAW"),
           ("Cornell", "CORNELL"), ("CUNY", "CUNY"), ("Dalhousie", "DALHOUSIE"), ("Dartmouth", "DARTMOUTH"), ("Davidson", "DAVIDSON"),
           ("Denison", "DENISON"), ("Drexel", "DREXEL"), ("Duke", "DUKE"), ("DuquesneUniversity", "DUQUESNEUNIVERSITY"), 
           ("Durham", "DURHAM"), ("Emmanuel", "EMMANUEL"), ("Emory", "EMORY"), ("Fairfield", "FAIRFIELD"), ("FisherCollege", "FISHERCOLLEGE"), 
           ("FIU", "FIU"), ("FloridaStateUniversity", "FLORIDASTATEUNIVERSITY"), ("Fordham", "FORDHAM"), 
           ("FranklinandMarshall", "FRANKLINANDMARSHALL"), ("GeorgeMason", "GEORGEMASON"), ("Georgetown", "GEORGETOWN"), 
           ("GeorgeWashington", "GEORGEWASHINGTON"), ("Glasgow", "GLASGOW"), ("Grinnell", "GRINELL"), 
           ("GroveCityCollege", "GROVECITYCOLLEGE"), ("Hamilton", "HAMILTON"), ("HartHouse", "HARTHOUSE"), ("Harvard", "HARVARD"), 
           ("HarvardLaw", "HARVARDLAW"), ("Haverford", "HAVERFORD"), ("HobartandWilliamSmith", "HOBARTANDWILLIAMSMITH"),
           ("IBADU", "IBADU"), ("IIUM", "IIUM"), ("JohnsHopkins", "JOHNSHOPKINS"), ("Kings", "KINGS"), 
           ("KwameNkrumahUniversityofScienceandTechnology", "KWAMENKRUMAHUNIVERSITYOFSCIENCEANDTECHNOLOGY"), ("LaVerne", "LAVERNE"), 
           ("Lehigh", "LEHIGH"), ("LoyolaMarymount", "LOYOLAMARYMOUNT"), ("LoyolaUniversityChicago", "LOYOLAUNIVERSITYCHICAGO"), 
           ("Maryland", "MARYLAND"), ("McGill", "MCGILL"), ("Middlebury", "MIDDLEBURY"), ("MIT", "MIT"),
           ("MoodyBibleInstitute", "MOODYBIBLEINSTITUTE"), ("Morehouse", "MOREHOUSE"), ("MorehouseCollege", "MOREHOUSECOLLEGE"),
           ("MountHolyoke", "MOUNTHOLYOKE"), ("Northeastern", "NORTHEASTERN"), ("Northwestern", "NORTHWESTERN"), 
           ("NotreDame", "NOTREDAME"), ("NYU", "NYU"), ("Odette", "ODETTE"), ("Ottawa", "OTTAWA"), ("Oxford", "OXFORD"), 
           ("Pace", "PACE"), ("PatrickHenry", "PATRICKHENRY"), ("Penn", "PENN"), 
           ("PrinceGeorgesCommunityCollege", "PRINCEGEORGESCOMMUNITYCOLLEGE"), ("Princeton", "PRINCETON"), 
           ("ProvidenceCollege", "PROVIDENCECOLLEGE"), ("Quakers", "QUAKERS"), ("QueensUniversity", "QUEENSUNIVERSITY"), ("RIT", "RIT"),
           ("Rochester", "ROCHESTER"), ("RPI", "RPI"), ("Rutgers", "RUTGERS"), ("SanJoseState", "SANJOSESTATE"), ("SantaClara", "SANTACLARA"),
           ("SimonFraserUniversity", "SIMONFRASERUNIVERSITY"), ("SimonsRockCollege", "SIMONSROCKCOLLEGE"), ("Skidmore", "SKIDMORE"), 
           ("Smith", "SMITH"), ("SouthCarolina", "SOUTHCAROLINA"), ("Spelman", "SPELMAN"), ("StAndrews", "STANDREWS"), ("Stanford", "STANFORD"), 
           ("StJohns", "STJOHNS"), ("StMarys", "STMARYS"), ("StonyBrookUniversity", "STONYBROOKUNIVERSITY"), ("Swarthmore", "SWARTHMORE"),
           ("Syracuse", "SYRACUSE"), ("TelAviv", "TELAVIV"), ("Temple", "TEMPLE"), ("TESU", "TESU"),
           ("TheCollegeofNewJersey", "THECOLLEGEOFNEWJERSEY"), ("Trinity", "TRINITY"), ("Tufts", "TUFTS"), ("Tulane", "TULANE"), 
           ("Tulsa", "TULSA"), ("UCDL&H", "UCDL&H"), ("UCLA", "UCLA"), ("UConn", "UCONN"), ("UMBC", "UMBC"), ("UniversityofAlaskaAnchorage", "UNIVERSITYOFALASKAANCHORAGE"),
           ("UniversityofAlbany", "UNIVERSITYOFALBANY"), ("UniversityofBritishColumbia", "UNIVERSITYOFBRITISHCOLUMBIA"), 
           ("UniversityofCalgary", "UNIVERSITYOFCALGARY"), ("UniversityofChicago", "UNIVERSITYOFCHICAGO"), ("UniversityofDelaware", "UNIVERSITYOFDELAWARE"),
           ("UniversityofDenver", "UNIVERSITYOFDENVER"), ("UniversityofGuelph", "UNIVERSITYOFGUELPH"), ("UniversityofHawaiiatManoa", "UNIVERSITYOFHAWAIIATMANOA"),
           ("UniversityofMassachusetts", "UNIVERSITYOFMASSACHUSETTS"), ("UniversityofMichigan", "UNIVERSITYOFMICHIGAN"), ("UniversityofMinnesota", "UNIVERSITYOFMINNESOTA"),
           ("UniversityofNewSouthWales", "UNIVERSITYOFNEWSOUTHWALES"), ("UniversityofNorthCarolina", "UNIVERSITYOFNORTHCAROLINA"), 
           ("UniversityofPittsburgh", "UNIVERSITYOFPITTSBURGH"), ("UniversityofSouthernCalifornia", "UNIVERSITYOFSOUTHERNCALIFORNIA"),
           ("UniversityofSydney", "UNIVERSITYOFSYDNEY"), ("UniversityofthePeople", "UNIVERSITYOFTHEPEOPLE"), ("UniversityofVermont", "UNIVERSITYOFVERMONT"),
           ("UniversityofVirginia", "UNIVERSITYOFVIRGINIA"), ("UniversityofWaterloo", "UNIVERSITYOFWATERLOO"), ("UTAustin", "UTAUSTIN"), ("Vassar", "VASSAR"),
           ("Villanova", "VILLANOVA"), ("WashingtonUniversityinStLouis", "WASHINGTONUNIVERSITYINSTLOUIS"), ("Wellesley", "WELLESLEY"), ("Wesleyan", "WESLEYAN"),
           ("Western", "WESTERN"), ("WilfredLaurierUniversity", "WILFREDLAURIERUNIVERSITY"), ("WilliamandMary", "WILLIAMANDMARY"), 
           ("Williams", "WILLIAMS"), ("WSCC", "WSCC"), ("YorkUniversity", "YORKUNIVERSITY"), ("Yale", "YALE")]

# Creates a df with all speakers seen in each school
all_debaters = []
for school, label in schools:
    school_debaters = []
    for year in range(2004, 2026):
        for html_path, school_name, year in get_debaters(school, year):

            with open(html_path, encoding="utf-8") as fp:
                soup = BeautifulSoup(fp, "html.parser")

            target_table = None

            for table in soup.find_all("table"):
                headers = [th.get_text(strip=True) for th in table.find_all("th")]
                if headers == ["ID", "Name", "Year on Team"]:
                    target_table = table
                    break

            if target_table is None:
                continue

            df = pd.read_html(StringIO(str(target_table)))[0]
            df["school"] = school_name
            df["initial_year_seen"] = year + 1 - df["Year on Team"]
            df["final_year_seen"] = year
            df = df.drop(columns=["ID", "Year on Team"])
            school_debaters.append(df)
    
    if not school_debaters:
        continue
    df = pd.concat(school_debaters, ignore_index=True)    
    df = (
        df.groupby(["Name", "school"], as_index=False)
          .agg({
              "initial_year_seen": "min",
              "final_year_seen": "max"
          })
    )
    df = df.drop_duplicates(keep="last")
    all_debaters.append(df)
    
    #df.to_csv(f"DebateRosters/{school}_debaters.csv", index=False)

all_debaters = pd.concat(all_debaters, ignore_index=True)
all_debaters = all_debaters.rename(columns={"Name": "debater"})
all_debaters = all_debaters.drop_duplicates(subset=["debater"], keep="last")
all_debaters.to_csv("Data_Curation_Data/Debater_Roster.csv", index=False)

In order to ensure that each of the names in the newly created Debate_Roster is a unique indiviudal, I utilized the Levenshtein algorithm to find the 10 nearest distances of each name, then I dropped all neighbors that were not from the same school nor within three years of their intial years nor withing three years of their last year. This was to make manual inspection easier. I then compared and corrected mistypes and nicknames throughout the roster. "Roster_Top_Lev_Matches.csv" stores the reamaining matches I inspected.

In [116]:
import Levenshtein
import pandas as pd

debate_roster = pd.read_csv("Data_Curation_Data/Debater_Roster.csv")

def normalize(name):
    return str(name).strip().lower().replace(",", "").replace(".", "")

debate_roster["debater_norm"] = debate_roster["debater"].apply(normalize)

TOP_N = 10
results = []

names = debate_roster["debater"].tolist()
norms = debate_roster["debater_norm"].tolist()
schools = debate_roster["school"].tolist()
initial_years = debate_roster["initial_year_seen"].tolist()
final_years = debate_roster["final_year_seen"].tolist()

for i in range(len(names)):
    name_i, norm_i, school_i = names[i], norms[i], schools[i]
    init_i, final_i = initial_years[i], final_years[i]

    scored = []
    for j in range(len(names)):
        if i == j:
            continue
        dist = Levenshtein.distance(norm_i, norms[j])
        scored.append((names[j], schools[j], initial_years[j], final_years[j], dist))

    scored.sort(key=lambda x: x[4])
    top_matches = scored[:TOP_N]

    for rank, (match_name, match_school, match_init, match_final, dist) in enumerate(top_matches, start=1):
        results.append({
            "name": name_i,
            "candidate_match": match_name,
            "school": school_i,
            "candidate_school": match_school,
            "initial_year_seen": init_i,
            "candidate_initial_year": match_init,
            "final_year_seen": final_i,
            "candidate_final_year": match_final,
            "rank": rank,
            "distance": dist,
        })

roster_dup_matches = pd.DataFrame(results)
roster_dup_matches = roster_dup_matches[roster_dup_matches["school"] == roster_dup_matches["candidate_school"]]
roster_dup_matches = roster_dup_matches[roster_dup_matches["initial_year_seen"] >= (roster_dup_matches["candidate_initial_year"] - 3)]
roster_dup_matches = roster_dup_matches[roster_dup_matches["final_year_seen"] <= (roster_dup_matches["candidate_final_year"] + 3)]
roster_dup_matches.to_csv("Data_Curation_Data/Roster_Top_Lev_Matches.csv", index=False)

These are the names I will drop as there is already a name to represent this debater in the set.

In [117]:
invalid_names = ["Zara Menon", "Trent Kannegieter", "Eva-Mare Quinones", "Sophia Mannina", "Chetan Hebur", "Chris Gooding", 
                 "Wes Mantingou", "Karl Mcghie", "Nithya Matcha", "Liam Class", "Vivian Li", "Jake Low", "Satoshi Tanaizu", "Rahul Gosain"
                 "Matthew Zinman", "Andrew Kamen", "Theresa Ng", "Syddarth Iyer", "Gordon Lev", "Juliana Vigorita", "Pete Falk", "Mark Siegel"
                 "Ben Frazer", "Edmund Saw", "Katie English", "Liam Rahav", "Drew Meseck", "Julia Melendi", "Annmaria Antony", "Ava Blakley",
                 "Nerary Flores", "Theodore Gercken2", "Jeff Gan", "Jon Shapiro", "Emma Buchsbaum", "Gabby Miller", "Eddy Xu", "Matt Schoenfeld",
                 "Adri Lorenzini", "Danny Blackman", "Ariela Stoylar", "Swapanthi Naguip", "Emma Rose Bienvenu", "Dan Kefer", "Dhurv Prakash",
                 "Leeana Skuby", "Pryanah Nadanasabesan", "Remi Riff", "Abraham Mhaidli", "Tim Colledge", "Cam Miranda-Radbord", "Mars",
                 "Mimi Alphonsus", "Alex Chang", "Jeffrey Yin", "Katelyn Keller", "Vasa Syomin", "Matt Gill", "L. Barra", "Michael Thgn", "Debi Ogunrinde",
                 "Alex Gecker", "Chris Bao", "Jamie Hintson", "Sam Mathews", "Shubhro Saha", "Rachel Figura", "Ryan Affallah", "Aaron Kahn", "EJ Kang",
                 "Molly Petchen", "Ben Field", "Danny Goynatsky", "Nick Onfrey", "Tim Ritter", "Alex Webster", "Camara Hudson", "Eve Fidler -",
                 "Shenandoah Worrel", "Shiwen Yeo"]
all_debaters = all_debaters[~all_debaters['debater'].isin(invalid_names)]
all_debaters.to_csv("Data_Curation_Data/Debater_Roster.csv", index=False)

In order to preserve data lost from entry errors, I utilized the Levenshtein algorithm to compare unidentified judges/debaters with the roster collected above. I compiled the top ten nearest matches and manually inspected the entries in order to identify errors and make adjustments. The below code creates the csvs for judging and debaters that I utilized and stored within "Lost_Judge_Top_Lev_Matches.csv" and "Lost_Debater_Top_Lev_Matchescsv".

In [118]:
all_rounds = pd.read_csv("Data_Curation_Data/Rounds.csv")
debate_roster = pd.read_csv("Data_Curation_Data/Debater_Roster.csv")
valid_names = set(debate_roster["debater"])

rounds_lost = all_rounds[~all_rounds["judge"].isin(valid_names)].copy()
print("Rounds Lost Before Levenshtein Adjustments: ", len(rounds_lost))

judge_names = debate_roster["debater"].tolist()
judges_lost = rounds_lost["judge"].value_counts()
TOP_N = 10
results = []

for lost_name, rounds_lost in judges_lost.items():
    scored = [(name, Levenshtein.distance(lost_name, name)) for name in judge_names]
    scored.sort(key=lambda x: x[1])

    top_matches = scored[:TOP_N]

    for rank, (name, dist) in enumerate(top_matches, start=1):
        results.append({
            "Lost Judge": lost_name,
            "Rounds Lost": rounds_lost,
            "Rank": rank,
            "Candidate Match": name,
            "Distance": dist,
        })

judge_match_suggestions = pd.DataFrame(results)
judge_match_suggestions.to_csv("Data_Curation_Data/Lost_Judge_Top_Lev_Matches.csv", index=False)

Rounds Lost Before Levenshtein Adjustments:  5638


In [119]:
all_rounds = pd.read_csv("Data_Curation_Data/Rounds.csv")
debate_roster = pd.read_csv("Data_Curation_Data/Debater_Roster.csv")

speaker_cols = ["speaker_one_name", "speaker_two_name", "opponent_one_name", "opponent_two_name"]
roster_names = debate_roster["debater"].tolist()
valid_names = set(roster_names)

combined = []
for col in speaker_cols:
    unmatched_mask = ~all_rounds[col].isin(valid_names)
    counts = all_rounds[col][unmatched_mask].value_counts()
    temp = counts.rename("rounds_lost").rename_axis("name").reset_index()
    temp["role"] = col
    combined.append(temp)

lost_rounds_all_speakers = pd.concat(combined, ignore_index=True)
lost_rounds_all_speakers = lost_rounds_all_speakers[["role", "name", "rounds_lost"]]

# run Levenshtein top matches for each unique unmatched name
TOP_N = 10
unique_lost_names = lost_rounds_all_speakers["name"].unique()

match_cache = {}
for lost_name in unique_lost_names:
    scored = [(roster_name, Levenshtein.distance(lost_name, roster_name))
              for roster_name in roster_names]
    scored.sort(key=lambda x: x[1])
    match_cache[lost_name] = scored[:TOP_N]

results = []
for _, row in lost_rounds_all_speakers.iterrows():
    lost_name = row["name"]
    for rank, (roster_name, dist) in enumerate(match_cache[lost_name], start=1):
        results.append({
            "lost_name": lost_name,
            "role": row["role"],
            "rounds_lost": row["rounds_lost"],
            "rank": rank,
            "candidate_match": roster_name,
            "distance": dist,
        })

match_suggestions = pd.DataFrame(results)
match_suggestions.to_csv("Data_Curation_Data/Lost_Debater_Top_Lev_Matches.csv", index=False)

The two files "Lost_Judge_Top_Lev_Matches.csv" and "Lost_Debater_Top_Lev_Matches.csv" allowed me to manually inspect the names for both judges and competitors and create the following dictionary. This catches many entering issues such as nicknames, misspellings, or middle name inclusion/exclusion that makes an individual's name different from the one registed on the APDA website. The following dictionary displays the changes I made:

In [7]:
COMMON_MISSPELLINGS = {
    "Jenni Pham": "Jenny Pham", "Jenni": "Jenny Pham", "Dominic Deramo": "Dominic DeRamo", "Joseph Rubas": "Julia Rubas", "Joey Rubas": "Julia Rubas", "Audrey J. Higley": "Audrey Higley", "Alessandro Perri": "Ale Perri", "Cece Szkutak": "CeCe Szkutak",
    "Nicholas Devito": "Nick DeVito", "Nick Devito": "Nick DeVito", "Kj Kniering": "KJ Kniering", "Ian Mcvann-Henkelmann": "Ian McVann-Henkelmann", "Ian Mcvann": "Ian McVann-Henkelmann", "Ian Mcvann-Henk Elmann": "Ian McVann-Henkelmann", "Ian Mcvann-Henke Lmann": "Ian McVann-Henkelmann",
    "Ian Mcvann-Henklemann": "Ian McVann-Henkelmann", "Ian Mcvann Henkelmann": "Ian McVann-Henkelmann", "Ian Mcvann Henkelma Nn": "Ian McVann-Henkelmann", "Izzy Jenkins": "Isabella Jenkins", "Max F. Neuman": "Max Neuman", "Max Neumann": "Max Neuman", "Naomi Mckenna": "Naomi McKenna",
    "Alejandro Franqui-Ferrer": "Alejandro Franqui", "Katherine St George": "Katherine St. George", "Claire Mcmahon Fishman": "Claire Fishman", "Eli Nelson": "Elijah Nelson", "Hannah Platter They/Them": "Hannah Platter", "Oliver Mccammon": "Oliver McCammon", "Eva Bruce She/Her": "Eva Bruce",
    "Andrew Kao*": "Andrew Kao", "Adam T Harrington": "Adam Harrington", "Ej Hermacinski": "EJ Hermacinski", "Ceci Granda-Scott": "Cecilia Granda-Scott", "Cam Chacon": "Cameron Chacon", "Sam Watkins": "Samuel Watkins", "Liz Esterbrook": "Elizabeth Esterbrook", "Katie Farrell": "Kathryn Farrell",
    "Wes Mcgovern": "Wes McGovern", "Aadhav Raviarasan": "Aadhavaarasan Raviarasan", "Yashas Mallikarju N": "Yashas Mallikarjun", "Gordon Mcneill": "Gordon McNeill", "Drew Harrington": "Andrew Harrington", "Alice Marchant": "Alice Merchant", "Vincent Kazella": "Vinny Kazz", 
    "Vinny Kazella": "Vinny Kazz", "Muzzi Godil": "Muzamil Godil", "Muzzi": "Muzamil Godil", "Stav Kanza": "Stav Kanza-Levi", "Aleisha Martinez Sandoval": "Aleisha Martínez", "Aleisha Martinez-Sandoval": "Aleisha Martínez", "Samhitha Duggirala": "Samhitha Duggurala",
    "Sheryar Fazal": "Sheryar Ahmad Fazal", "Sam Widell": "Sam Widwell", "Saif Elkhodor": "Saif El Khodor", "Adi Jayakrishnan":"Aditya Jayakrishnan", "Madeleine Watson": "Maddie Watson", "Giuseppe Dimassa": "Giuseppe DiMassa", "Benjamin Grimes": "Benji Grimes", "Cassie Fitts": "Cassandra Fitts",
    "Abby Fechisso": "Abigail Fechisso", "Madeline Cheshire": "Maddy Cheshire", "Tamhid Islam": "Tahmid Islam", "Janul De Silva": "Janul de Silva", "Claire Fraise": "Claire Frase", "Allison Ross": "Ally Ross", "Zach Lemonides": "Zachary Lemonides", "Alexander Gerber": "Alex Gerber",
    "Gabe Lomonaco": "Gabriel Lomonaco", "Aidan Hollinger Miles": "Aidan Hollinger-Miles", "Zachary Malek": "Zack Malek", "Theo Miranda-Zellnik": "Theo Miranda-Zellink", "Becca Shields": "Rebecca Shields", "Gabriel Ritter": "Gabe Ritter", "Gabe Lomonaco": "Gabriel Lomonaco", "Gabe Lomanaco": "Gabriel Lomonaco",
    "Eva Quinones": "Eva-Marie Quinones", "Will Arnesen": "William Arnesen", "Da'Von Boyd": "Davon Boyd", "Anne Motovillof": "Anne Motoviloff", "Nikos Efthymiadis": "Nikolaos Efthymiadis", "Will Zeng": "William Zeng", "Nokutenda Zuze": "Noku Zuze", "Joanne Bai": "Joanna Bai",
    "Dillion Ma": "Dillon Ma", "Omesh": "Omesh Dhar Dwivedi", "Andrew Bellows": "Andrew Bell", "Nat Puapattanakajorn": "Nat Puappatankajorn", "Matt Feng": "Matthew Feng", "Tosca Neuma Nn": "Tosca Neumann", "Bartholomew Kaminski": "Bart Kaminski", "Benjamin Scherzer": "Ben Scherzer",
    "Alexander Purn": "Alex Purn", "Eddie Siderenko": "Eddie Sidorenko", "Katie Zhang": "Kate Zhang", "Peter Kladais": "Peter Kladias", "Jaiden Hassan": "Jaiden Hasan", "Muku Madzivire": "Mukudzeiishe Madzivire", "Pranav Gargipati": "Pranav Garigipati", "Sonam Wangchuk": "Sonam Tenzin Wangchuk",
    "Max Sheremeta": "Maxwell Sheremeta", "Aditya Ram": "Adi Ram", "Robert Neilson": "Robert Nielsen", "Leandro Guevara": "Leandro Guevara-Neyra", "Timothy Goggin": "Tim Goggin", "Matthew Rubenstein": "Matt Rubenstein", "Tori Reiz": "Tori Reisz", "Devyani :": "Devyani Goel", "Bella Sorial": "Isabella Sorial",
    "Ale": "Ale Perri", "Aiza": "Aiza Nygman", "Ari Hanh": "Ari Hahn", "Maxwell Sheremata": "Maxwell Sheremeta", "Harry Carr": "Harrison Carr", "Samantha Wing": "Samantha Wong", "Will Donnely": "Will Donnelly", "Shruti Narayanabha Tla": "Shruti Narayanabhatla", "Alexa Ross": "Ally Ross",
    "Sami Cuaresma": "Samara Cuaresma", "Xiao-Ke Lu": "Xiao-ke Lu", "Nikki Schuldt": "Niki Schrift", "Christopher O’Keeffe":	"Christopher O'Keeffe", "Brendan Mcdermott": "Brendan McDermott", "Nicholas Lim": "Nicolas Lim", "Joon Sohn": "Joonpyo Sohn", "Dylan Gyauch-Lewis": "Dylan Gyauch Lewis",
    "Alexandra Dischler": "Alex Dischler", "Siddharth Ramanathan": "Sid Ramanathan", "Matt Rohn": "Matthew Rohn", "Ina Ralhakar": "Ina Rahalkar", "Ong Unjiwatana": "Ong Unjitwattana", "Rob Nielsen": "Robert Nielsen", "Cornelia": "Cornelia Hsieh", "Mitch Mullen": "Mitchell Mullen",
    "Kaley Kathleen Alexandre-Burke": "Kaley Alexandre-Burke", "Angier Li": "Angier Lei", "Gabbi Schilcusky": "Gabbi Shilcusky", "Mehul Agrawal": "Mehul Agarwal", "Mindy Hupsen": "Mindy Huspen", "Andrew Montieth": "Andrew Monteith", "Fee Pelz-Sharpe": "Fiona Petz-Sharpe", "Preston Johnston": "Preston Johnson", 
    "Sonam": "Sonam Tenzin Wangchuk", "Andrew Liang": "Andrew Laing", "Reca Safarti": "Reca Sarfati", "Will Meyer": "William Meyer", "Anthony Peña": "Anthony Pena", "Rishven K Pravin": "Rishven Pravin", "Gregory Gentile": "Greg Gentile", "Matt Simons": "Matthew Simons", "Addie Gill T": "Addie Gill", 
    "Micahel Ryter": "Michael Ryter", "Martin Gazsner": "Martin Gaszner", "Ari Han": "Ari Hahn", "Nick Cathcart": "Nicolas Cathcart", "Andew Monteith": "Andrew Monteith", "Liberty": "Liberty Prieb", "Cody Mcmanus": "Cody McManus", "Foula Christopoulos": "Foula Christopolous",
    "Michelle Teicher": "Michelle Tiecher", "Hannah Owens Pierre": "Hannah Owens-Pierre", "William Howard-Driemeier": "William Hallward-Driemeier", "Ej Kang": "Eunjae Kang", "William Huang": "Will Huang", "Kyuryeon Kim": "Khuryeon Kim", "Pètra De Beer": "Petra de Beer", "Erica Morelli": "Erica Morellli",
    "Arushi Agrawal": "Arushi Agarwal", "Uszee Mckoy": "UsZee Mckoy", "Efrain Thomas Ortiz": "Efrain Ortiz", "Vivienne Montiero": "Vivienne Monteiro", "Benjamin Cortez": "Ben Cortez", "Yva": "Yva Totchum", "Elena Blake Leeds": "Elena Leeds", "Tammy Yamile Leon Molina": "Tammy Yamile León Molina",
    "Sophia Tyrrell Knott": "Sophia Tyrrell-Knott", "Rennie": "Rennie Lee", "Sumanth M": "Sumanth Mahalingam", "Jaice": "Jaice Williamson", "Josh Sampson": "Joshua Sampson", "Jeffery Gao": "Jeffrey Gao", "Andres Mendoza Casas": "Andres Mendozas Casas", "Matt Lee": "Mathew Lee",
    "Dan Perez": "Daniel Perez", "Claire Mchahon Fishman": "Claire Fishman", "Cecilia Szkutak": "CeCe Szkutak", "Jela Shriver": "Jela Shiver", "Matthew Franco": "Matt Franco", "Max Kornfield": "Max Kornfeld", "Egor Cherniuk": "Egor Chernyuk", "Catie Macauley": "Catie Mccauley",
    "Will Choi": "William Choi", "Carlos Irrisari": "Carlos Irisarri", "Charlie Mclarnon": "Charlie McLarnon", "Dilay Kalinoglu": "Dilay Kolinoglu", "Clemente Nicado-Yelmene": "Clemente Nicado Yelmene", "Natalie Keim": "Nat Keim", "Joseph Mcgroarty": "Joseph McGroarty", "Ananya Ganish": "Ananya Ganesh",
    "Sumanth Mahalinga M": "Sumanth Mahalingam", "Marcel": "Marcel Cato", "Arielle Gallagos": "Arielle Gallegos", "Khy Stubblefield": "Khylan Stubblefield", "Zander": "Zander Jeinthanuttkanont", "Neftalí Reynoso": "Neftali Reynoso", "Sheryar": "Sheryar Ahmad Fazal", "Teddy Jack": "TJ Jack",
    "Rafael Rodriguez": "Rafael Rodriguez Alvarez", "Will Logue": "William Logue", "Adi Jayakrishna N": "Aditya Jayakrishnan", "Gaurav": "Gaurav Gawankar", "Joe Brennan": "Joeseph Brennan", "Alina Ziying Su": "Alina Su", "Jiwhan Moon": "JiWhan Moon", "Jack Reeed": "Jack Reed", "Arianna Hellman T": "Arianna Hellman",
    "Paola Apolinari O": "Paola Apolinario", "Lizzie Walters": "Lizzie Waters", "Julia Shepard": "Julia Shephard", "Dominic !": "Dominic DeRamo", "Gian Luigi Zaninelli": "GianLuigi Zaninelli", "Vara Mathiylaakan": "Vara Mathiyalakan", "Nat Nichanun Puapattanakajorn": "Nat Puappatankajorn", 
    "Sid Chakravarthy": "Siddhaarth Chakravarthy", "Susan Mcharris": "Susan McHarris", "Sydney C": "Sydney Cook", "Sarah Cobau-": "Sarah Cobau", "Ryan Geary .": "Ryan Geary", "Spike": "Spike King", "Andrew Harrington .": "Andrew Harrington", "Ry-Ry Geary": "Ryan Geary", "Alex Uy-Tioco": "Alexandra Uy-Tioco",
    "James Cox-Donovan": "James Donovan", "Mikala Parnell": "Mikala Pernell", "Rishika Deshide": "Rish Deshide", "Violet Whitimire": "Violet Whitmire", "Alejandro Franqui Ferrer": "Alejandro Franqui", "Eva Marie Quinones": "Eva-Marie Quinones", "Sania Ifran": "Sania Irfan", "Riya Singh": "Rhea Singh",
    "Lindsey Gradow Ski": "Lindsey Gradowski", "Rishve N Pravin": "Rishven Pravin", "Pranav Garigip Ati": "Pranav Garigipati", "Ong Unjiwata Na": "Ong Unjitwattana", "William Hallward-Dri Emeier": "William Hallward-Driemeier", "Oscar Cloutier": "Oscar Cloutier Potter", "Oscar Clouti Er Potter": "Oscar Cloutier Potter",
    "Domi Nic Dera Mo": "Dominic DeRamo", "Eftychia  Christodoulou": "Eftychia Christodoulou", "Melina Piatta-Chayan": "Melina Piatti-Chayan", "Sophia Winner-": "Sophia Winner", "Giuseppe Di Massa": "Giuseppe DiMassa", "Gianluigi Zaninelli": "GianLuigi Zaninelli", "Kathleen Mcintyre": "Kathleen McIntyre",
    "Ana Carolina Marques Perez": "Ana Marques Perez", "Kaya Panchalinga M": "Kaya Panchalingam", "Aadhav Raviarasan": "Aadhavaarasan Raviarasan", "Aadhavaarasa N Raviarasan": "Aadhavaarasan Raviarasan", "Keshav Malik Kapoor": "Keshav Kapoor", "Abby Hill": "Abigail Hill", "Ceci Granda Scott": "Cecilia Granda-Scott",
    "Cecilia Granda Scott": "Cecilia Granda-Scott", "Emma Listgarte N": "Emma Listgarten", "Zander Jeinthanuttkano Nt": "Zander Jeinthanuttkanont", "Zander Jeinthannatkanont": "Zander Jeinthanuttkanont", "Abdullah Mejjalid": "Abdullah Mejjallid", "Abhishek Amit Shah": "Abhishek Shah",
    "Aidan Duran Rey": "Adrian Duran", "Aidan Gilles": "Aidan Gillies", "Aiden Shannon": "Aidan Shannon", "Aislinn O'Brian": "Aislinn O'Brien", "Akash": "Akash Shivakumaar", "Alexander Gordon": "Alex Gordon", "Alexander Schramm": "Alex Schramm", "Alexandra Uy-Tico": "Alexandra Uy-Tioco", 
    "Allison Chan": "Alison Chan", "Alvaro Marin-Garcia": "Alvaro Marin Garcia", "Amala Kari": "Amala Karri", "Amira Butani": "Amira Bhutani", "An Lahn Le": "An-Lanh Le", "An Lanh Le": "An-Lanh Le", "Ana Marquez Perez": "Ana Marques Perez", "Andrew Harington": "Andrew Harrington",
    "Andrew Rozenbilt": "Andrew Rozenblit", "Anna Keternos": "Anna Ketrenos", "Ariana Hellman": "Arianna Hellman", "Audrey J Higley": "Audrey Higley", "Audri Bhowmick": "Audri Bhomick", "Ava Schneiburg": "Ava Schneiberg", "Awsam Boaubid":"Awsam Bouabid", "Zach Braunstein":"Zachary Braunstein",
    "Alex Elsrodt": "Alex Elstrodt", "Elena Lille": "Elena Lill", "Zimo Tracy Ge": "Zimo-Tracy Ge", "Zhouai Joann Yu": "Zhouai Joann", "Zan Rosen": "Zan Rozen", "William Shachar": "Will Shachar", "Tori Fekete": "Victoria Fekete", "Clemente Yelmene Nicado": "Clemente Nicado Yelmene", "Alex Elsdrodt": "Alex Elstrodt",
    "Sophia Mason": "Sophie Mason", "Zara Mermon": "Zara Memon", "Zachary Brown": "Zach Brown", "Yuqiu Rachel Liu": "Yuqian Li", "Yanni Trimiklionitis": "Yanni Trimikliniotis", "Yana Sharifulli Na": "Yana Sharifullina", "Xiao Ke Lu": "Xiao-ke Lu", "Wilson Shen": "Wilson Chen", "Wilson Cheung": "Winson Cheung",
    "William-Hallward-Driemeier": "William Hallward-Driemeier", "William Hallward-Dreiemeir": "William Hallward-Driemeier", "William Hallward Driemeier": "William Hallward-Driemeier", "Vita Raskevičiūtė": "Vita Raskeviciute", "Viraj": "Viraj Nautiyal", "Vikram Balasubramania N": "Vikram Balasubramanian", "Vihini Gunaseker A": "Vihini Gunasekera",
    "Vignesh Mehrotra": "Vighnesh Mehrotra", "Viet Thé Phan": "Viet Phan", "Vasily Syomin": "Vasiley Syomin", "Vasa Syomin": "Vasiley Syomin", "Varsha Sripadha M": "Varsha Sripadham", "Trey Garcia Schartz": "Trey Garcia-Schartz", "Tilly Swanson": "Matilda Swanson", "Thomas Ii Hyun Kim": "Thomas Kim",
    "Theodore Gerken": "Theodore Gercken", "Theodore Gercken2": "Theodore Gercken", "Theodore": "Theodore Gercken", "Theodor Gerken": "Theodore Gercken", "Tanya": "Tanya Chatterjee", "Tai Hendricks": "Tai Henrichs", "Pranav Garigapati": "Pranav Garigipati", "Pranav Garigiapti": "Pranav Garigipati",
    "Pranav Gpt": "Pranav Garigipati", "Oscar Cloutier-Potter": "Oscar Cloutier Potter", "Oscar Potter": "Oscar Cloutier Potter", "Szilveszter Palvolgyi": "Szilvester Palvolgyi", "Ollie Saunder S": "Olivia Saunders", "Ollie Saunders": "Olivia Saunders", "Nicole Salinas Reyes": "Nicole Salinas-Reyes", "Nichanun Puapattanakajom": "Nat Puappatankajorn",
    "Nat Puapattankajor N": "Nat Puappatankajorn", "Muzammil Godil": "Muzamil Godil", "Muzzamil Godil": "Muzamil Godil", "Maxwell Hurowitz": "Max Hurowitz", "Sankey Bhalotia": "Sanket Bhalotia", "Sanket Bhlatoia": "Sanket Bhalotia", "Santiago Cantu": "Santi Cantu", "Sara Zdancewi Cz": "Sara Zdancewicz", "Susanna Kirtzler": "Susanna Kritzler", 
    "Sulley Ho": "Sullivan Ho", "Srishti Ghosh": "Srishti Gosh", "Sophie Rose Fetter": "Sophie Fetter", "Sophia Torres Da Cruz": "Sophia Torres da Cruz", "Siddharth Ramanatha N": "Sid Ramanathan", "Shruthi Bharath Kumar": "Sruthi Bharath Kumar", "Siddharth Ramanathan": "Sid Ramanathan", "Siddharth Ramanathan": "Sid Ramanathan",
    "Scott Santella": "Scott Santaella", "Sav Stackhouse": "Savannah Stackhouse", "Samuel Arneson": "Sam Arnesen", "Samantha Pryzbisiki": "Samantha Przybisiki", "Sam Sulzinksy": "Sam Sulzinsky", "Sam Melcher": "Samuel Melcher", "Sajan Mehrota": "Sajan Mehrotra", "Saif El-Khodor": "Saif El Khodor",
    "Ryyaan Sheikh": "Rayan Sheikh", "Ryan Tiedmann": "Ryan Tiedemann", "Ryan Tiedema Nn": "Ryan Tiedemann", "Ryan G": "Ryan Tiedemann", "Romina Lillolari": "Romina Lilollari", "Rocco Spinozzi-D’Andrea": "Rocco Spinozzi-D'Andrea", "Robert Redine": "Robert Rendine", "Robert Neilsen": "Robert Nielsen", "Rita Mayevsyaka": "Rita Mayevskaya",
    "Rish Pravin": "Rishven Pravin", "Ricky Kiamliev": "Ricky Kiamilev", "Reya Kalolwala": "Reyya Kalolwala", "Rachel Robbin": "Rachel Robin", "Pranav Sasikumae": "Pranav Sasikumar", "Pragyna Yerramalli": "Pragnya Yerramalli", "Petra De Beer": "Petra de Beer", "Peregrine Beckett": "Perry Beckett",
    "Patrik Dugan": "Patrick Dugan", "Patrick Mccarthy": "Patrick McCarthy", "Thibaut Juneja": "Shanyu Thibaut Juneja", "Sam Rowher": "Sam Rohwer", "Samuel Rohwer": "Sam Rohwer", "Sam Duggirala": "Samhitha Duggurala", "Sam Arneson": "Sam Arnesen", "Rocco Spinozzi D'Andrea": "Rocco Spinozzi-D'Andrea", "Nitin Kumar Saidha": "Nitin Kumar",
    "Nikita Chakraborty": "Nikhita Chakraborty", "Nicole Kagain": "Nicole Kagan", "Nicolas Parra": "Nick Parra", "Marcelo Parra": "Marcelo Rodriguez Parra", "Nico Hortiguera": "Nicolas Hortiguera", "Nicholas Mccarthy": "Nicholas McCarthy", "Nichanun Puapattanakajorn": "Nat Puappatankajorn", "Nathaniel": "Nathaniel Yoon",
    "Natali Saraf": "Natali Sarraf", "Nat Pupattanakajor N": "Nat Puappatankajorn", "Nash Reibe": "Nash Riebe", "Mohammad Sarker": "Mohammed Sarker", "Misimi Sanni": "Mismi Sanni", "Michelle Li": "Michelle Liu", "Michael Schermerhorn": "Mike Schermerhorn", "Michael Hanson": "Michael Hansen", "Kevin Hammil": "Kevin Hammill", 
    "Micah": "Micah Kawecki", "Mckinley Cherrier": "McKinley Cherrier", "Maxwell Taborrok": "Maxwell Tabarrok", "Max Wiener": "Maxwell Weiner", "Nico Llorente": "Nico Llorente Valin", "Travis Hunsburger": "Travis Hunsberger", "Ian Gate": "Ian Gates", "Mounisha Anumolo": "Mounisha Anumolu",
    "Karan Kuppa-Ape": "Karan Kappa-Apte", "Alex Hellinghausen": "Alexandra Hellinghausen", "Mac O’Hara": "Mac O'Hara", "Amarachi Alozie": "Amarchi Alozie", "Bev Soriano": "Bey Soriano", "Max Tabarrok": "Maxwell Tabarrok", "Max Sheremata Gu": "Maxwell Sheremeta", "Matthew Rubensten": "Matt Rubenstein",
    "Matthew Rubenstei N": "Matt Rubenstein", "Mathew Moreno": "Matthew Moreno", "Mario Aguire": "Mario Aguirre", "Mariana Icaza-Diaz": "Mariana Icaza Diaz", "Manuel Machorro Gomez Pezuela": "Manuel Machorro", "Madison Damien": "Madison Damian", "Madeleine Eichhorn": "Madeleine Eichorn", "Maddy Chesire": "Maddy Cheshire",
    "Maddie Nagel": "Maddie Nagle", "Mackenzi Tran": "Mackenzie Tran", "Mac O'Hare": "Mac O'Hara", "Mac Hayes": "Mac Hays", "Mabel Reiger": "Mabel Rieger", "Lê Quang Trịnh": "Quang Trinh", "Lydia Vlastro": "Lydia Vlasto", "Luke Joel Shankar": "Joel Shankar", "Lizzie Mccord": "Lizzie McCord",
    "Lizz Kim": "Liz Kim", "Lindsey Gradowsk I": "Lindsey Gradowski", "Lindsey Gradow": "Lindsey Gradowski", "Lily Levin": "Lilly Levin", "Liliana D’Aguiar": "Liliana D'Aguiar", "Lesley Munenyasha Machimbidza": "Lesley Machimbidza", "Leandro Guevara Neyra": "Leandro Guevara-Neyra", "Khylan Stubblefiel D": "Khylan Stubblefield",
    "Khylan Stubblefi Eld": "Khylan Stubblefield", "Ibtihal Gassem": "Ibithal Gassem", "Gabriel Frank-Mcpheter": "Gabriel Frank-McPheter", "Yaroslav Opansyuk": "Yaroslav Opanasyuk", "Zachary Fedyk": "Zach Fedyk", "Sam Arneson": "Sam Arnesen", "Kensington Speer": "Kensington Spear",
    "Kcale Teevan": "Cale Teevan", "Sunint Bundra": "Sunint Bindra", "Alice Merolli": "Alice Meroli", "Anshul Khakhar": "Anshul Khakar", "Anushri Swivedi": "Anushri Dwivedi", "Ariana Sharifi": "Ariane Sharifi", "Avrey Li": "Avery Li", "Caitlyn Jaeyeon Kim": "Caitlyn Kim",
    "Catie Macaulay": "Catie Mccauley", "Cayleigh Solderholm": "Cayleigh Soderholm", "Cecelia Szkutak": "CeCe Szkutak", "Cees Armstrong": "Coen Armstrong", "Christian Seskosan": "Christian Sekosan", "Claire Fennel": "Claire Fennell", "Colin Mac Hays": "Mac Hays", "Cody Magnus": "Cody McManus",
    "Curtis Lee": "Kurtis Lee", "Nicholas Milan": "Nicolas Millan", "Kayla Chen": "Kayla Chan", "Karan Kuppa-Apte": "Karan Kappa-Apte", "Jullia Chanda": "Jullian Chanda", "Julia W.": "Julia Wang", "Judy Jiang": "Judie Jiang", "Joshua Neudorf": "Josh Neudorf", "Joshua Kretchmer": "Josh Kretchmer", "Josh Tandiono": "Joshua Tandiono",
    "Josh Josheph": "Josh Joseph", "Joseph Bilotta": "Joe Bilotta", "Jorge Arturo Ramirez": "Jorge Ramirez", "Joe Billotta": "Joe Bilotta", "Jillian Elkins-Brumwell": "Jillian Elkins", "Jess Wang": "Jessica Wang", "Jenn Tran": "Jennifer Tran", "David Morales Lam": "David Morales",
    "David Ruvagaa": "David Ruvaga", "Dea Karemeti": "Dea Karameti", "Desmund Hui": "Desmond Hui", "Devesh Kodani": "Devesh Kodnani", "Dhafer Muhammed": "Dhafer Muhammad", "Dhruva Sumeshwar": "Dhruva Someshwar", "Dom Passafium E": "Dom Passafiume", "Dominic Digioia": "Dom Digioia", "Drew Meseck": "Drew Mesek",
    "Eden Rowe": "Eden Row", "Elijajh Nelson": "Elijah Nelson", "Elisa Gonzales": "Elisa Gonzalez", "Elizabeth Burkle": "Elizabeth Buerkle", "Emilio Stuart-Alb An": "Emilio Stuart-Alban", "Emily Zheng": "Emily Zhang", "Emma Jean Hermacinski": "EJ Hermacinski", "Erika": "Erika McCague",
    "Ethan James Mcminn": "Ethan James McMinn", "Ethan Mcminn": "Ethan James McMinn", "Gautam Ramasamy": "Gautum Ramasamy", "Erica Dinapoli": "Erica DiNapoli", "Elliot Mokski": "Elliott Mokski", "Ethan Rosenbaum": "Evan Rosenbaum", "Ethan Shurburg": "Ethan Shurberg", "Ezza Tariq": "Ezzah Tariq",
    "Fabi Ortez": "Fabbi Ortez", "Gabi Cunningha M": "Gabi Cunningham", "Genesis Lopez De Leon": "Genesis Lopez", "Gorbo Shilcusky": "Gabbi Shilcusky", "Gordon Mcneil": "Gordon McNeill", "Grace Flyyn": "Grace Flynn", "Grace Mctigue": "Grace McTigue", "Sophia Tyrell-Knott": "Sophia Tyrrell-Knott",
    "Alex Gellman-Beer": "Alex Gellman", "Greg Gentil": "Greg Gentile", "Griffin Badlamente": "Griffin Badalamente", "Grishma Baraugh": "Grishma Baruah", "Gwendolyn Havern": "Gwen Havern", "Habiba Mbugua": "Habib Moody", "Harrison Lavelle": "Harisson Lavelle", "Helen W": "Helen Wu", "Himnashu Padnani": "Himanshu Padnani",
    "I'Yanna Jones": "I'yanna Jones", "Ioannis Kryiakou": "Ioannis Kyriakou", "Christopher Vanderpool": "Chris Vanderpool", "Isa Irvine": "Isabel Irvine", "Jack Manimalco": "Jack Maniscalco", "Jake Fenster": "Jacob Fenster", "Jake Wasinger": "Jacob Wasinger", "James Eiferman": "James Eigerman",
    "Jane Metzinger": "Jane Mentzinger", "Janul Da Silva": "Janul de Silva", "Jarden Lenn": "Jared Lenn", "Jay Lewis": "James Lewis", "Jeffery Sheng": "Jeffrey Sheng", "Phil Maniscalco": "Phillip Maniscalo", "Jehan Chanmugan": "Jehan Chanmugam", "Karina Wugang": "Karina Wuwang", "Fletcher Calcagano": "Fletcher Calcagno", 
    "Lem Yu": "Lemuel Yu", "Sándor Erik Lorange": "Sándor Lorange", "Sandor Lorange": "Sándor Lorange", "Reverand Sandor Lorange": "Sándor Lorange", "Eden Rowe": "Eden Row", "Alexander Bennett": "Alex Bennett", "Rafi Chowdurry": "Rafi Chowdhury", "Vedant Kerjariwal": "Vedant Kejariwal",
    "Elizabeth A Chen": "Elizabeth Chen", "Alessandro Perri": "Ale Perri", "Dan Rochon": "Daniel Rochon", "Joshua Cohen": "Josh Cohen", "Simran Verma-Singh": "Simran Singh", "Leemah B.": "Leemah Bisht", "Dixsheta Muralikrishnan": "Dixie Muralikrishnan", "Gabe Salgado": "Gabriel Salgado",
    "Danny Lee": "Denny Lee", "Sándor Erik Lorange":"Sándor Lorange", "Samuel Slack": "Sam Slack", "Caroline Sagristano": "Caroline Sagristino", "Bella Campbell": "Annabella Campbell", "Nathan Sears": "Natt Sears", "Nicolas Llorente Valin": "Nico Llorente Valin", "Hannah Platter": "Hannah Platter", "Eunisa Lu": "Eunisa Liu", "Eva Bruce ":  
    "Eva Bruce", "Devank Agarwal": "Devansh Agarwal", "Kathryn Perrone": "Kate Perrone", "Zoe Savoy Rose": "Zoe Rose", "Anisah Colon": "Anisah Colón", "Katie Mae Ryan": "Katherine Ryan", "Kate Santarelli": "Katie Santarelli", "Nico Llorente-Valin": "Nico Llorente Valin", "Ícaro Teixeira": "Icaro Teixeira",
    "Jess Mcelroy": "Jessica Mcelroy", "Genevieve Savage": "Geneveive Savage", "Kiran Subramaniam": "Kiran Subramanian", "Sanket Bhaloti": "Sanket Bhalotia", "Pranav Garigipat I": "Pranav Garigipat", "Emma Listgarten V": "Emma Listgarten", "Jubayer Hamid": "Jubayer Ibn Hamid", "Anshika Agarwal": "Anshika Agrawal", "Lucia Bronfm An": "Lucia Bronfman",
    "Vita Raskeciute":"Vita Raskeviciute", "Maneet Mehta N": "Maneet Mehta", "Chih-Hsien Sam Liu": "Sam Liu", "Flo Arnado N": "Flo Arnado", "Mindy Leblanc": "Mindy LeBlanc", "Charvi .": "Charvi Bhayana", "Charvi Bhayana .": "Charvi Bhayana", "Damian Vladimiroff": "Damian Vladmiroff", "Kaylah Costa": "Kayla Costa", "Daniel Leizerman": "Dan Leizerman",
    "Rush Patel": "Rushabh Patel", "Danny Welden": "Daniel Welden", "Nyankpani Kesson Abdul-Quddus": "Nyankpani Kesson Abdul-Quddud", "Zara Menon": "Zara Memon", "Trent Kannegieter": "Trent Kannekieter", "Sophia Mannina": "Sophia Manning", "Chetan Hebur":"Chetan Hebbur", "Chris Gooding": "Chris Golding", "Wes Mantingou": "Wes Matingou", 
    "Karl Mcghie": "Karl McGhie", "Nithya Matcha": "Nitya Matcha", "Liam Class":"Liam Glass", "Vivian Li": "Vivian Liu", "Jacob Emont":"Jacob Emount", "Jake Low": "Jake Lowe", "Satoshi Tanaizu": "Satoshi Yanaizu", "Rahul Gosain": "Rahul Gossain", "Matthew Zinman": "Matthew Zinnman", "Andrew Kamen": "Andrew Kaman", "Theresa Ng": "Teresa Ng", 
    "Syddarth Iyer": "Siddarth Iyer", "Gordon Lev": "Gordon Lew", "Juliana Vigorita": "Juliana Vigorito", "Pete Falk": "Peter Falk", "Mark Siegel": "Marc Siegel", "Ben Frazer": "Ben Frazier", "Edmund Saw": "Edmund Shaw", "Katie English": "Kate English", "Liam Rahav": "Liam Rajav", "Julia Melendi": "Julian Melendi", "Annmaria Antony": "Annamaria Antony",
    "Ava Blakley": "Ava Blakeley", "Nerary Flores": "Merary Flores", "Jeff Gan": "Jeffrey Fan", "Jon Shapiro": "Jonathan Shapiro", "Emma Buchsbaum": "Emma Buschbaum", "Gabby Miller": "Gabriella Miller", "Victoria Sliwa": "Victoria Silwa", "Eddy Xu": "Edward Xu", "Matt Schoenfeld": "Matthew Schoenfield", "Adri Lorenzini": "Adriana Lorenzini",
    "Danny Blackman": "Daniel Blackman", "Ariela Stoylar": "Ariela Stolyar", "Swapanthi Naguip": "Swapanthi Nagupally", "Emma Rose Bienvenu": "Emma Rose", "Dan Kefer": "Daniel Kefer", "Dhurv Prakash": "Dhruv Prakash", "Leeana Skuby": "Leanna Skulby", "Pryanah Nadanasabesan": "Priyanha Nadanasabesan", "Remi Riff": "Raymond Rif", "Abraham Mhaidli": "Abraham Maidlhi",
    "Tim Colledge": "Timothy Colledge", "Cam Miranda-Radbord": "Cameron Miranda-Radbord", "Mars": "Mars He", "Mimi Alphonsus": "Miriam Alphonsus", "Alex Chang": "Alexander Chang", "Jeffrey Yin": "Jeffery Yin", "Katelyn Keller": "Kate Keller", "Matt Gill": "Matthew Gill", "L. Barra": "Lauren Barra", "Michael Thgn": "Michael Thng", "Debi Ogunrinde": "Debi Ogurrinne",
    "Alex Gecker": "Alexandra Gecker", "Chris Bao": "Christopher Bao", "Jamie Hintson": "Jamie Hinston", "Sam Mathews": "Samuel Matthews", "Shubhro Saha": "Shubrho Saha", "Rachel Figura": "Rachel Rose-Figura", "Ryan Affallah": "Ryan Atallah", "Aaron Kahn": "Aaron Khan", "EJ Kang": "Eunjae Kang", "Molly Petchen": "Molly Petchenik", "Ben Field": "Benjamin Field",
    "Danny Goynatsky": "Daniel Goynatsky", "Nick Onfrey": "Nick Confrey", "Tim Ritter": "Timothy Ritter", "Alex Webster": "Alexandra Webster", "Camara Hudson": "Camara Stokes Hudson", "Eve Fidler -": "Eve Fidler", "Shenandoah Worrel": "Shenan Worrel", "Shiwen Yeo": "Shi Wen Yeo"
}

Now that the entry errors are fixed we can examine out new lost rounds and see how it has improved.

In [8]:
all_rounds = pd.read_csv("Data_Curation_Data/Rounds.csv")
debate_roster = pd.read_csv("Data_Curation_Data/Debater_Roster.csv")
valid_names = set(debate_roster["debater"])

# Replaces the names with common errors
debater_cols = ["judge", "speaker_one_name", "speaker_two_name", "opponent_one_name", "opponent_two_name"]
for col in debater_cols:
    all_rounds[col] = all_rounds[col].replace(COMMON_MISSPELLINGS)

# Sorts to rounds lost due to unidentified judge
rounds_lost = all_rounds[~all_rounds["judge"].isin(valid_names)].copy()
#rounds_lost.to_csv("rounds_Lost_Post_Lev.csv", index=False)
print("Rounds Lost After Levenshtein Adjustments: ", len(rounds_lost))

Rounds Lost After Levenshtein Adjustments:  4332


From this we see that we have improved the lost rounds considerable, but many rounds remain dropped. The next way to retain rounds will be to create an analysis who each individual has partnered with and see what schools those individuals are from. The majority of debaters debate with a partner from their own school, especially when attending their first tournaments. It is a safe bet that the majority of tournaments most debaters attend is with a partner of their own school. To do this, lets first analyze what rows were dropped so that we might see the debaters that were unrecognized.

While examing the Levenshtein distance files, I realized that minor parsing issues remained so the following pattern cleans for these remaining issues in the following sections.

In [9]:
# Patterns viewed in the Levenshtein distance files that need to be corrected before preceeding
unclean_patterns = [
    r"\*",
    r"\(R\)",
    r"\(",
    r"\)",
    r"She/Her",
    r"He/Him",
    r"They/She",
    r"He/They",
    r"They/Them",
]

The following creates a method that merges the "Rounds.csv" with a valid file containing debaters from the offical roster, adding information on the judge of the round if information of them is stored in the roster passed.

In [10]:
def add_judges_info(debate_roster):
    debate_rounds_df = pd.read_csv("Data_Curation_Data/Rounds.csv")
    
    # Performs cleaning and replacment
    for pat in unclean_patterns:
            debate_rounds_df["judge"] = debate_rounds_df["judge"].str.replace(pat, "", regex=True)
    debate_rounds_df["judge"] = debate_rounds_df["judge"].str.strip()
    debate_rounds_df["judge"] = debate_rounds_df["judge"].replace(COMMON_MISSPELLINGS)
    valid_debaters = pd.read_csv(debate_roster)
    
    judge_school_df = debate_rounds_df.rename(columns={"judge": "debater"})
    judge_school_df = judge_school_df.sort_values("year")
    valid_debaters_sorted = valid_debaters.sort_values("initial_year_seen")
    
    # First pass: match the affiliation that had already started by the round's year
    judge_school_df = pd.merge_asof(
        judge_school_df,
        valid_debaters_sorted,
        left_on="year",
        right_on="initial_year_seen",
        by="debater",
        direction="backward",
    )
    
    # Second pass: for rows still unmatched, fall back to each debater's earliest known affiliation
    earliest_debaters = (
        valid_debaters
        .sort_values("initial_year_seen")
        .drop_duplicates(subset="debater", keep="first")
        .set_index("debater")
    )
    
    still_missing = judge_school_df["school"].isna()
    
    judge_school_df.loc[still_missing, "school"] = (
        judge_school_df.loc[still_missing, "debater"].map(earliest_debaters["school"])
    )
    judge_school_df.loc[still_missing, "initial_year_seen"] = (
        judge_school_df.loc[still_missing, "debater"].map(earliest_debaters["initial_year_seen"])
    )
    judge_school_df.loc[still_missing, "final_year_seen"] = (
        judge_school_df.loc[still_missing, "debater"].map(earliest_debaters["final_year_seen"])
    )
    
    judge_school_df = judge_school_df.rename(columns={
        "debater": "judge",
        "school": "judge_school",
        "initial_year_seen": "judge_initial_year",
        "final_year_seen": "judge_last_competed"
    })
    return judge_school_df

The following updates stores round data after this addition, which will now contain all the rows where the judge name can be correlated with a name on the "Debater_Roster.csv" file post the cleaning of remaning problematic patterns and the misspelling corrections. The file is created by merging the two datasets and stored into "rounds_Before_Judge_Restoration".

In [11]:
judges_df = add_judges_info("Data_Curation_Data/Debater_Roster.csv")
judges_df.to_csv("Data_Curation_Data/rounds_Before_Judge_Restoration.csv", index=False)

The following similarly constructs a method that merges the a passed version of round data with a passed file containing debater roster information. The method adds information on the speakers and opponents in the round if information of them is stored in the roster passed.

In [12]:
def add_debater_info(debater_roster, rounds_df, label):
    id_judge_df = pd.read_csv(rounds_df)
    
    speaker_cols = ["speaker_one_name", "speaker_two_name", "opponent_one_name", "opponent_two_name"]
    
    for col in speaker_cols:
        for pat in unclean_patterns:
            id_judge_df[col] = id_judge_df[col].str.replace(pat, "", regex=True)
        id_judge_df[col] = id_judge_df[col].str.strip() 
        id_judge_df[col] = id_judge_df[col].replace(COMMON_MISSPELLINGS)    
    debate_roster = pd.read_csv(debater_roster)
    
    roster_lookup = debate_roster.drop_duplicates(subset="debater").set_index("debater")
    valid_names = set(roster_lookup.index)
    
    name_columns = {
        "speaker_one_name": "speaker_one",
        "speaker_two_name": "speaker_two",
        "opponent_one_name": "opponent_one",
        "opponent_two_name": "opponent_two",
    }
    
    lost_rounds = {}
    
    for name_col, prefix in name_columns.items():
        cleaned = id_judge_df[name_col].replace(COMMON_MISSPELLINGS)
    
        unmatched_mask = ~cleaned.isin(valid_names)
        lost_rounds[prefix] = cleaned[unmatched_mask].value_counts()
    
        id_judge_df[name_col] = cleaned
        id_judge_df[f"{prefix}_school"] = cleaned.map(roster_lookup["school"])
        id_judge_df[f"{prefix}_initial_year"] = cleaned.map(roster_lookup["initial_year_seen"])
        id_judge_df[f"{prefix}_last_competed"] = cleaned.map(roster_lookup["final_year_seen"])
    
    
    indiv_speaker_s1 = lost_rounds["speaker_one"]
    indiv_speaker_s2 = lost_rounds["speaker_two"]
    indiv_speaker_o1 = lost_rounds["opponent_one"]
    indiv_speaker_o2 = lost_rounds["opponent_two"]
    
    df_speaker_ids = id_judge_df.copy()
    
    df_speaker_ids["speaker_hybrid_status"] = df_speaker_ids["speaker_one_school"] != df_speaker_ids["speaker_two_school"]
    df_speaker_ids["opponent_hybrid_status"] = df_speaker_ids["opponent_one_school"] != df_speaker_ids["opponent_two_school"]
    
    df_speaker_ids.to_csv(label, index=False)
    return id_judge_df

In [13]:
id_judge_df = add_debater_info("Data_Curation_Data/Debater_Roster.csv", "Data_Curation_Data/rounds_Before_Judge_Restoration.csv", "Data_Curation_Data/rounds_Before_Debate_Restoration.csv")

In order to examine the unidentified debaters, I will create a file that lists the number of partners a debater has attended tournaments with along with the school of the identified debater. This will help to reconstruct debaters that have never been entered into the official roster. For example, Abby Sweeney has been to 24 tournaments with a George Washington partner but is not on the roster for any school. We can assume she is from George Washington.

In [14]:
partner_pairs = []
partner_pairs.append(id_judge_df[["speaker_one_name", "speaker_two_school", "year"]]
                      .rename(columns={"speaker_one_name": "debater", "speaker_two_school": "partner_school"}))
partner_pairs.append(id_judge_df[["speaker_two_name", "speaker_one_school", "year"]]
                      .rename(columns={"speaker_two_name": "debater", "speaker_one_school": "partner_school"}))
partner_pairs.append(id_judge_df[["opponent_one_name", "opponent_two_school", "year"]]
                      .rename(columns={"opponent_one_name": "debater", "opponent_two_school": "partner_school"}))
partner_pairs.append(id_judge_df[["opponent_two_name", "opponent_one_school", "year"]]
                      .rename(columns={"opponent_two_name": "debater", "opponent_one_school": "partner_school"}))

all_partner_pairs = pd.concat(partner_pairs, ignore_index=True).dropna(subset=["debater", "partner_school"])

all_partner_pairs_unmatched = all_partner_pairs[
    ~all_partner_pairs["debater"].isin(valid_names)
].copy()

partner_school_summary_unmatched = (
    all_partner_pairs_unmatched.groupby("debater")["partner_school"]
    .agg(lambda schools: sorted(set(schools)))
    .reset_index()
    .rename(columns={"partner_school": "distinct_partner_schools"})
)
partner_school_summary_unmatched["num_distinct_partner_schools"] = (
    partner_school_summary_unmatched["distinct_partner_schools"].apply(len)
)

year_range_unmatched = (
    all_partner_pairs_unmatched.groupby("debater")["year"]
    .agg(First_year_Seen="min", Last_year_Seen="max")
    .reset_index()
)

partner_school_summary_unmatched = pd.merge(
    partner_school_summary_unmatched, year_range_unmatched, on="debater", how="left"
)
partner_school_summary_unmatched = partner_school_summary_unmatched.sort_values(
    "num_distinct_partner_schools", ascending=False
)
#partner_school_summary_unmatched.to_csv("partner_school_summary_unmatched.csv", index=False)

partner_school_counts_unmatched = (
    all_partner_pairs_unmatched.groupby(["debater", "partner_school"])
    .agg(rounds=("year", "size"), First_year_Seen=("year", "min"), Last_year_Seen=("year", "max"))
    .reset_index()
    .sort_values(["debater", "rounds"], ascending=[True, False])
)
partner_school_counts_unmatched = partner_school_counts_unmatched.rename(columns={"rounds": "tournaments"})
partner_school_counts_unmatched.to_csv("Data_Curation_Data/Partner_School_Counts.csv", index=False)

From the "Partner_School_Counts.csv" document I created new rows to be appened to the "Debate_Roster.csv". The rows are detailed below and saved into a new file names "Recovered_Debaters.csv" in total I recovered 987 debaters not listed in the debate roster, but who had competed in debate rounds between Fall 2019 and Spring 2026. Additionally, the "Debate_Roster.csv" and "Recovered_Debaters.csv" files are combined into one file names "Combined_Debaters.csv".

In [15]:
recovered_debaters = [ 
    {"debater": "Alex Chaparro", "school": "Johns Hopkins",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Anthony Bragoli", "school": "University of Virginia",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Clara Yang", "school": "Williams",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Isabela Eliassen", "school": "University of Massachusetts",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Jana Kassem", "school": "University of Chicago",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Joey Deleone", "school": "Smith",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Judy Liu", "school": "Georgetown",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Matt Seitz", "school": "Rutgers",   "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Jack Anschultz", "school": "University of Massachusetts",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Isaiah Minter", "school": "University of Massachusetts",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Imaan Chaudhry", "school": "Smith",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Ian Gates", "school": "University of Virginia",   "initial_year": 2019, "last_year_seen": 2019},  {"debater": "Hikaru Hayakawa", "school": "Tufts",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Henry Combs", "school": "University of Chicago",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Germalysa Ferrer", "school": "Princeton",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Gabriel Mock", "school": "University of Massachusetts",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Ellie Park", "school": "Boston University",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Eleni Antoniades", "school": "Haverford",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Dylan Greenspan", "school": "Binghamton University",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Daniel Coughlin", "school": "Binghamton University",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Avina Sharma", "school": "Rutgers",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Jonathan Morse", "school": "University of Massachusetts",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Kevin Hammill", "school": "Fordham",   "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Samridhi Parasrampuria","school": "Boston University",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Tania Acsinte", "school": "University of Chicago",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Mila Maglov", "school": "Brandeis",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Katrina Deng", "school": "Boston University",   "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Julia Sicard","school": "University of Massachusetts",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Jude Hoag", "school": "Boston University",   "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Joyce Choi", "school": "Georgetown",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Jane Kassemk", "school": "University of Chicago",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Kiran Das-Goel", "school": "Smith",   "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Lily Kwak", "school": "Columbia",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Malak Hajiyeva", "school": "University of Massachusetts",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Matt Pekor", "school": "University of Pittsburgh",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Maura Baker", "school": "Haverford",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Michael Hoffman", "school": "George Washington",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Michael Okpoti", "school": "Boston University",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Pranav Sundar", "school": "Haverford",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Reece Danzis", "school": "Rutgers",   "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Ronnie Hokett", "school": "Binghamton University",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Sam Brown", "school": "Rutgers",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Sefat Uddin Samee", "school": "Tufts",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Jay Mathur", "school": "Maryland",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Jay Philbrick", "school": "Brown",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Jeffrey Stein", "school": "Columbia",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Jennie Fan", "school": "Penn",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Jennifer Lin", "school": "Boston University",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Joe Clark", "school": "Boston University",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Jonathan Iacovacci", "school": "Maryland",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Yun Zhang", "school": "Bates",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Vedant Kejariwal", "school": "Boston University",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Tyler Swartz", "school": "Rutgers",   "initial_year": 2019, "last_year_seen": 2020}, {"debater": "Taijah Chavis", "school": "Boston University",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Jack Tajmajer", "school": "Brown",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Jackson Kramp", "school": "Lehigh",   "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Jacob Weinberg", "school": "Fordham",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Jack Shapiro", "school": "Maryland",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Jacoby Sypher", "school": "George Washington",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Jacqueline Wang", "school": "Brandeis",   "initial_year": 2019, "last_year_seen": 2020}, {"debater": "Jade Ye", "school": "Bentley University",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Jai Kovvuri", "school": "Pace",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Jaime Colon", "school": "Northeastern",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Jake Bohman", "school": "Swarthmore",   "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Jake Lesser", "school": "University of Pittsburgh",   "initial_year": 2019, "last_year_seen": 2020}, {"debater": "Jamie Davis", "school": "George Washington",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Jana Jandal Alrifai", "school": "Tufts",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Jaqueline Rizzi", "school": "Boston University",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Jaren Friesen", "school": "Boston University",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Jason Alder", "school": "University of Massachusetts",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Jay Kim", "school": "William and Mary",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Ian Bass", "school": "Rutgers",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Igal Sultanov", "school": "CUNY",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Ilana Gellman", "school": "University of Chicago",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Ilina Logani", "school": "Columbia",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Iman Obargi", "school": "NYU",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Irene Kim", "school": "Yale",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Iris Kim", "school": "Haverford",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Isaac Pedersen", "school": "Northeastern",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Isabella Franklin", "school": "NYU",   "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Ishan Patel", "school": "Georgetown",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Israel Pierre", "school": "University of Chicago",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Issa Sadamoto", "school": "Stanford",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Iveena Mukherjee", "school": "Penn",   "initial_year": 2026, "last_year_seen": 2026}, {"debater": "Jack Hunt", "school": "Villanova",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Jack Shapiro", "school": "Maryland",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Hyunsoo Lee", "school": "UMBC",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Hugo Hinze", "school": "Harvard",   "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Hong Meng Yam", "school": "Stanford",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Holly Dickinson", "school": "Smith",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Hiyanshi Patel", "school": "Maryland",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Hermione Cordeiro Larkin", "school": "Johns Hopkins",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Henry Ren", "school": "Georgetown",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Henry Lei", "school": "Swarthmore",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Hemanth Asirvatham", "school": "Harvard",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Helen Phi", "school": "Brown",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Hayley Yeung",  "school": "University of Chicago",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Hayk Kibarian", "school": "American",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Harris Agha", "school": "Williams",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Harisson Lavelle", "school": "University of Virginia",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Hannah To", "school": "Princeton",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Hannah Messaye", "school": "Amherst",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Haneul Shin", "school": "Boston University",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Hamza Kalim", "school": "Bates",   "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Billy Donoso", "school": "Tufts",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Blake Lukens", "school": "Bentley University",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Caitlin Han", "school": "Tufts",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Chelsea Lugat", "school": "Binghamton University",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Christopher Chiu", "school": "Villanova",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "David Choi", "school": "Penn",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "David Timmerman", "school": "Binghamton University",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Destiny Eversole", "school": "Wellesley",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Diyor Kamolov", "school": "Columbia",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Drew Bennison", "school": "University of Virginia",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Ethan Lin", "school": "Amherst",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Guangxinyang Deng", "school": "University of Chicago",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Gwen Havern", "school": "Haverford",   "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Gwen Stearns", "school": "Johns Hopkins",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Habikah Baldeh", "school": "Johns Hopkins",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Hagan Werner", "school": "George Washington",   "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Hai Tran", "school": "University of Massachusetts",   "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Hailey Demars", "school": "Stanford",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Hailey Lorence", "school": "American",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Haitong Du", "school": "George Washington",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Aaron Kopew", "school": "Rutgers",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Abhishek Girish", "school": "NYU",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Adam Usmanov", "school": "University of Pittsburgh",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Adeolu Ajayi", "school": "Columbia",   "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Aidan Kuk", "school": "Johns Hopkins",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Alex Gellman", "school": "William and Mary",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Ambika Kandasemy", "school": "Johns Hopkins",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Anna Izyumova", "school": "Princeton",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Ayushi Singh", "school": "University of Massachusetts",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Ben Luo", "school": "University of Chicago",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Ben Xu", "school": "Tufts",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Nicole Berglund", "school": "University of Massachusetts",   "initial_year": 2020, "last_year_seen": 2020},
    {"debater": "Nicolas Howayeck", "school": "University of Massachusetts",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Nico Cavalluzzi", "school": "Boston University",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Nathan Olski", "school": "University of Virginia",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Maya Hoffman", "school": "Georgetown",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Martin Lapczyk", "school": "NYU",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Margaret Macgillivray", "school": "Williams",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Makal Matthews", "school": "Prince George's Community College",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Maia Harrison", "school": "Princeton",   "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Lindsay Lake", "school": "Brown",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Lena Sernoff", "school": "NYU",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Lani Craig", "school": "William and Mary",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Kylie Kim", "school": "Boston University",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Keevon Thomas", "school": "Temple",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Kate Perrone", "school": "University of Virginia",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "John Lynch", "school": "William and Mary",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Stephanie Lee", "school": "Johns Hopkins",   "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Trevor Cope", "school": "University of Massachusetts",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Yi Huang", "school": "Maryland",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Sophia Cruz", "school": "Grinnell",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Shayan Reza", "school": "University of Massachusetts",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Serkute Abebe", "school": "Columbia",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Seha Karabacak", "school": "Tufts",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Santosh Gontla", "school": "Davidson",   "initial_year": 2026, "last_year_seen": 2026}, {"debater": "Sam Wing", "school": "Binghamton University",   "initial_year": 2020, "last_year_seen": 2020},
    {"debater": "Sam Hano", "school": "University of Massachusetts",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Sal Conte", "school": "Bentley University",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Ruhaan Chopra", "school": "Fordham",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Ronald Taylor", "school": "Grinnell",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Robert Summers Berger", "school": "Carnegie Mellon",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Rahi Patel", "school": "University of Massachusetts",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Raegan Arroyo", "school": "Fordham",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Onder Kilinc", "school": "University of Chicago",   "initial_year": 2020, "last_year_seen": 2020},
    {"debater": "Gaston Aime", "school": "Tufts",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Gaurav Bagur", "school": "Brandeis",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Gavin Roulett", "school": "William and Mary",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Gena Rising", "school": "Binghamton University",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Genesis Lopez", "school": "University of Massachusetts",   "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Geneveive Savage", "school": "Northeastern",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "George Harrison", "school": "University of Chicago",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Georgedaniel Dixon", "school": "Amherst",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Gianna Bruno", "school": "Brandeis",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Gianna Naulivou", "school": "University of Massachusetts",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Gladwin An", "school": "Rutgers",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Govind Menon", "school": "Carnegie Mellon",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Grace Shamer", "school": "William and Mary",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Grace Wang", "school": "Fordham",   "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Grace Wu", "school": "Boston University",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Graham Sagel", "school": "Tufts",   "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Dylan Wallerstein", "school": "University of Massachusetts",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Elizabeth Qiao", "school": "Johns Hopkins",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Florence Nelson", "school": "Prince George's Community College",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Francisca Wijaya", "school": "Wesleyan",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Frank Chiu", "school": "Brown",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Frank Parizek", "school": "William and Mary",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Gabe Cronin-Golomb", "school": "Smith",   "initial_year": 2019, "last_year_seen": 2020}, {"debater": "Gabriel Uceda-Sosa", "school": "Columbia",   "initial_year": 2025, "last_year_seen": 2025},
    {"debater": "Gabriel Young", "school": "George Washington",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Gabriela Roznawska", "school": "Grinnell",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Gaby Ivanova", "school": "University of Chicago",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Garvin Kim", "school": "University of Chicago",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Emily Dale", "school": "Princeton",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Eric Parrilla", "school": "University of Chicago",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Ethan Quinn", "school": "Temple",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Eva Bingham", "school": "George Washington",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Evan Forrest", "school": "University of Massachusetts",   "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Evan Michaels", "school": "American",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Evelyn Chen", "school": "Northeastern",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Evyn Appel", "school": "University of North Carolina",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Ewa Tryniszewski", "school": "Georgetown",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Ezzat Abouleish", "school": "Yale",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Faith Ale", "school": "Rutgers",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Fiona Yang", "school": "Maryland",   "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Flavia Maria Galeazzi", "school": "Brown",   "initial_year": 2022, "last_year_seen": 2023}, {"debater": "James Coppersmith", "school": "Columbia",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Jason Ramdeo", "school": "William and Mary",   "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Jemima Williams", "school": "Princeton",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Emmanuel Yamba", "school": "George Washington",   "initial_year": 2019, "last_year_seen": 2020}, {"debater": "Hector Hernandez", "school": "Tufts",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Jailam Hutton", "school": "Temple",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Jake Macnelly", "school": "Boston College",   "initial_year": 2024, "last_year_seen": 2025},
    {"debater": "Dylan Safai", "school": "Williams",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Elise Greene", "school": "University of Massachusetts",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Elizabeth Rengifo", "school": "Fordham",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Elizabeth Rice", "school": "Bentley University",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Elizaveta Bakhtina", "school": "William and Mary",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Erin Hunter", "school": "University of Massachusetts",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Ethan Brown", "school": "Hamilton",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Erin Howard", "school": "Wellesley",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Ethan Carter", "school": "NYU",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Ethan Dubinsky", "school": "Fordham",   "initial_year": 2022, "last_year_seen": 2025}, {"debater": "Ethan Knox", "school": "Penn",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Eric Trueswell", "school": "University of Massachusetts",   "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Eunisa Liu", "school": "Maryland",   "initial_year": 2019, "last_year_seen": 2020}, {"debater": "Gabriel Salgado", "school": "University of Pittsburgh",   "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Hailey Baker", "school": "Fordham",   "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Jaelyn Perez", "school": "University of Massachusetts",   "initial_year": 2024, "last_year_seen": 2025},
    {"debater": "Jake Grande", "school": "George Washington",   "initial_year": 2019, "last_year_seen": 2020}, {"debater": "James Elmore", "school": "William and Mary",   "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Jake Sledge", "school": "Princeton",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Ella Bullock-Papa", "school": "Stanford",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Ella Scott", "school": "Washington University in St. Louis",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Ellie Berenson", "school": "William and Mary",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Elliot Jones", "school": "Wellesley",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Emanuel Yamba", "school": "George Washington",   "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Emily Flood", "school": "Harvard",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Emily Herstine", "school": "Temple",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Emily Kuchuk", "school": "Rutgers",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Emma Mcdonough", "school": "Northeastern",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Emma Mcfall", "school": "Brown",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Emma Smith", "school": "Georgetown",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Emmanuel Abay", "school": "Davidson",   "initial_year": 2026, "last_year_seen": 2026}, {"debater": "Erica Fisher", "school": "University of Pittsburgh",   "initial_year": 2020, "last_year_seen": 2020},
    {"debater": "Erica Otte", "school": "Maryland",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Erika Vasquez-Rivas", "school": "University of Chicago",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Erin Converse", "school": "Johns Hopkins",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Erin Howard", "school": "Wellesley",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Eden Row", "school": "Columbia",   "initial_year": 2020, "last_year_seen": 2021}, {"debater": "Edona Cosovic", "school": "Harvard",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Edward Frazer", "school": "Yale",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Edward Orji", "school": "Northeastern",   "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Edward Yang", "school": "NYU",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Efe Alpay", "school": "Brown",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Eileen Qiu", "school": "Brandeis",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Ej Beck", "school": "University of Chicago",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Elaina Craig", "school": "Fordham",   "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Elaine Wang", "school": "Brown",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Elena Prisament", "school": "MIT",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Elizabeth Chen", "school": "CUNY",   "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Elizabeth Janes", "school": "Lehigh",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Elizabeth Kean", "school": "Georgetown",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Elizabeth Kim", "school": "Boston University",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Elizabeth Morison", "school": "Fordham",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Alan Pham", "school": "Smith",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Carley Medeiros", "school": "Johns Hopkins",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Derek Zhen", "school": "Washington University in St. Louis",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Dev Patel", "school": "Maryland",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Devin Mullen", "school": "Rutgers",   "initial_year": 2019, "last_year_seen": 2019},{"debater": "Devin Pracar", "school": "Carnegie Mellon",   "initial_year": 2019, "last_year_seen": 2019},{"debater": "Dheeraj Pasikanti", "school": "George Washington",   "initial_year": 2025, "last_year_seen": 2025},{"debater": "Dhruv Kohli", "school": "University of Chicago",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Diana Gothong", "school": "University of Chicago",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Diego Estrada Adame", "school": "University of Chicago",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Dixie Muralikrishnan", "school": "Swarthmore",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Diyor Kamalov", "school": "Columbia",   "initial_year": 2023, "last_year_seen": 2024},
    {"debater": "Do Nguyen Tung", "school": "Princeton",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Dovran Babayev", "school": "Fordham",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Drew Morehead", "school": "Brown",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Drew Thomas", "school": "University of Chicago",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Dung Nguyen", "school": "Johns Hopkins",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "E'Niah Preston", "school": "Rutgers",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Ebenezer Appiah", "school": "Brown",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Dean Walters", "school": "University of Pittsburgh",   "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "David Dai", "school": "Harvard",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "David Borawski", "school": "University of Massachusetts",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Daria Mitri", "school": "Rutgers",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Dany Matar", "school": "Washington University in St. Louis",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Danilo Garcia", "school": "CUNY",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Clarissa Dias", "school": "Amherst",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Brian Forgue", "school": "University of Massachusetts",   "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Avi Konduri", "school": "Columbia",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Alexandra Hellinghausen","school": "Fordham",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Achuthan Panikath", "school": "University of Massachusetts",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Abielle Ahn", "school": "Rutgers",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Jennifer Tran", "school": "Rutgers",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Jenny Chen", "school": "Johns Hopkins",   "initial_year": 2021, "last_year_seen": 2022}, {"debater": "Jeremiah Harrington", "school": "Bates",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Jeremy Evans", "school": "Brandeis",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Jessica Johnson", "school": "Smith",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Jessica Mcelroy", "school": "Rutgers",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Jessie Zhou", "school": "Smith",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Jewel Thomas", "school": "University of Virginia",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Jiahao Guo", "school": "Georgetown",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Joey Arniel", "school": "Bates",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Jielu Yu", "school": "University of Chicago",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Jillian Elkins", "school": "Brandeis",   "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Jimmy Galvin", "school": "University of Virginia",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Jimmy Goranov", "school": "University of Virginia",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Jimmy Pham", "school": "Swarthmore",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Jin Huang", "school": "University of Chicago",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Jingchen Peng", "school": "NYU",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Jinge Cao", "school": "Boston University",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Jocelyn Gao", "school": "Rutgers",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Joe Anderson", "school": "Penn",   "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Joe Maalouf", "school": "Hamilton",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Joe Patti", "school": "Columbia",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Joey Cano", "school": "Amherst",   "initial_year": 2026, "last_year_seen": 2026}, {"debater": "Joey Juul God", "school": "Northeastern",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Johann Lindberg", "school": "William and Mary",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "John Ablonczy", "school": "University of Massachusetts",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "John Godwin", "school": "University of Virginia",   "initial_year": 2021, "last_year_seen": 2022}, {"debater": "John Polito", "school": "Georgetown",   "initial_year": 2024, "last_year_seen": 2025},
    {"debater": "Jonah Wunder", "school": "American",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Jonathan Pinelli", "school": "Tufts",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Jordan Medved", "school": "Brandeis",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Joseph Dey", "school": "University of Chicago",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Josephine Kuo", "school": "Tufts",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Josh Book", "school": "Columbia",   "initial_year": 2019, "last_year_seen": 2019},{"debater": "Josh Cohen", "school": "Tufts",   "initial_year": 2021, "last_year_seen": 2024}, {"debater": "Josh Joseph", "school": "Brandeis",   "initial_year": 2019, "last_year_seen": 2020}, 
    {"debater": "Joshua Ehizibolo", "school": "Maryland",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Julia Brous", "school": "Hamilton",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Julia Dietrich", "school": "University of Pittsburgh",   "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Julia Karabolli", "school": "UMBC",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Julio Cordero", "school": "CUNY",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Juno Tantipipatpong", "school": "Brown",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Justin Palus", "school": "Rutgers",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Justin Shillingford", "school": "Fordham",   "initial_year": 2023, "last_year_seen": 2025},
    {"debater": "Justin Woo", "school": "Brown",   "initial_year": 2020, "last_year_seen": 2021}, {"debater": "Kanika Mehra", "school": "Maryland",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Kanisha Harrell", "school": "Johns Hopkins",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Karan Makkar", "school": "Haverford",   "initial_year": 2019, "last_year_seen": 2020}, {"debater": "Karen Kao", "school": "Smith",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Karim Zohdy", "school": "Brown",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Kate Davis", "school": "University of Chicago",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Kate Selig", "school": "Stanford",   "initial_year": 2020, "last_year_seen": 2020},
    {"debater": "Kate Stover", "school": "Fordham",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Katelyn Rickert", "school": "Georgetown",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Katherine Ryan", "school": "Carnegie Mellon",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Kathy Piperno", "school": "Rutgers",   "initial_year": 2019, "last_year_seen": 2020}, {"debater": "Katie Santarelli", "school": "Johns Hopkins",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Kaydra Hopkins", "school": "Northeastern",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Kayla Costa", "school": "Johns Hopkins",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Kayla Morrison", "school": "Brown",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Mansi Bahl", "school": "University of Massachusetts",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Margo Mandell", "school": "Georgetown",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Nisha Athrey", "school": "Georgetown",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Noah Lennon", "school": "Binghamton University",   "initial_year": 2019, "last_year_seen": 2019},  {"debater": "Raaid Khan", "school": "Lehigh",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Sahil Gaba", "school": "Georgetown",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Shayan Raza", "school": "University of Massachusetts",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Sylvie Mahoro", "school": "Bates",   "initial_year": 2020, "last_year_seen": 2020},
    {"debater": "Daniel Welden", "school": "Pace",   "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Daniel Rochon", "school": "George Washington",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Daniel Leizerman", "school": "Bates",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Daniel Haskell", "school": "Bowdoin College",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Dan Leizerman", "school": "Bates",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Damian Vladmiroff", "school": "Boston University",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Curtis Mcmackin", "school": "Maryland",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Curtis Everett", "school": "Harvard",   "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Cristiana Ramos", "school": "Boston University",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Cormac Kimberly", "school": "Yale",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Conor Bifulco", "school": "NYU",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Colten Edelman", "school": "Brown",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Claire Paul", "school": "Boston University",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Chuck Foreman", "school": "Binghamton University",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Christopher Owen", "school": "William and Mary",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Christopher Lynch", "school": "Dartmouth",   "initial_year": 2022, "last_year_seen": 2022},   
    {"debater": "Christina Wang", "school": "Bates",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Christian Sekosan", "school": "Binghamton University",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Christian Gentolia", "school": "Fordham",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Carlos Freyre", "school": "William and Mary",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Chapin Fish", "school": "Fordham",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Charitha Vennapusa", "school": "University of Chicago",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Charles Tufenkji", "school": "Amherst",   "initial_year": 2026, "last_year_seen": 2026}, {"debater": "Chase Bezonsky", "school": "University of Massachusetts",   "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Chelsea Long", "school": "Brown",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Cheryl Minde", "school": "Smith",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Chris Francis Anto", "school": "Johns Hopkins",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Carlo Puca", "school": "University of Chicago",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Caleb Shook", "school": "University of Pittsburgh",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Brendan Heaney", "school": "Binghamton University",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Blake Mcneely", "school": "American",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Bianca Rozario", "school": "Boston University",   "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Ben Scharr-Weiner", "school": "Tufts",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Ben Robertson", "school": "Brandeis",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Ben Fligelman", "school": "Haverford",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Ayanna Williams", "school": "William and Mary",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Ava Peifer", "school": "William and Mary",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Askar Mirza", "school": "Rutgers",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Ashna Guha", "school": "University of Massachusetts",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Asher Moss", "school": "Villanova",   "initial_year": 2022, "last_year_seen": 2023},
    {"debater": "Anvi Shettigar", "school": "Boston University",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Anna Harshman", "school": "William and Mary",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Angie Guo", "school": "Brandeis",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Amarchi Alozie", "school": "Bates",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Allison Buehler", "school": "American",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Alex Hotzpaffel", "school": "William and Mary",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Akkshansh Bagga", "school": "Amherst",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Aiden Dowd", "school": "University of Virginia",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Advait Ganapathy", "school": "Harvard",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Adrian Kim", "school": "Rutgers",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Adrian Brown", "school": "Johns Hopkins",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Karan Kappa-Apte", "school": "Bates",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Karen Li", "school": "Wellesley",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Kayleigh Hernandez", "school": "Washington University in St. Louis",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Keith Do", "school": "Wesleyan",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Kelvin Powell", "school": "Columbia",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Kenneth Chen", "school": "Harvard",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Kiran Das Goel", "school": "Smith",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Malika Buribayeva", "school": "Lehigh",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Marcelo Gonzales", "school": "George Washington",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Max Markel", "school": "William and Mary",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Mia Flaherty", "school": "American",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Mikael Zarett", "school": "Stanford",   "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Miles Katz", "school": "NYU",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Naomi O'Meara", "school": "Rutgers",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Noura Ag", "school": "Brandeis",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Owen Meroski", "school": "University of Massachusetts",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Philip Surendran", "school": "Dartmouth",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Rishi Hazra", "school": "Harvard",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Ryan Zheng", "school": "Boston College",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Salma Sheikh", "school": "Rutgers",   "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Sarah Loewecke", "school": "Fordham",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Simar Soni", "school": "Penn",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Syed Azan Ali", "school": "Georgetown",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Tushar Dalmia", "school": "University of Chicago",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Uriah Colegrove", "school": "Columbia",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Victoria Hagen", "school": "Boston University",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Vik Georgieva", "school": "Wesleyan",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "William Pirone", "school": "University of Chicago",   "initial_year": 2023, "last_year_seen": 2025},
    {"debater": "Hana Hussain", "school": "Lehigh",   "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Icaro Teixeira", "school": "Williams",   "initial_year": 2025, "last_year_seen": 2026}, {"debater": "Isabella Rocco", "school": "Johns Hopkins",   "initial_year": 2019, "last_year_seen": 2020}, {"debater": "Jack Keating", "school": "William and Mary",   "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Jada Badillo", "school": "Smith",   "initial_year": 2019, "last_year_seen": 2022}, {"debater": "Kenyatta Heavlow", "school": "University of Massachusetts",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Kevin Cho", "school": "Rutgers",   "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Kevin Khadavi", "school": "University of Chicago",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Kevin Xue", "school": "Penn",   "initial_year": 2024, "last_year_seen": 2026}, {"debater": "Khadeeja Qureshi", "school": "Bates",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Khanh Doan", "school": "American",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Khuslen Tulga", "school": "Hamilton",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Kingston Lew", "school": "MIT",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Krishin Wadhwani", "school": "Carnegie Mellon",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Krishma Gewali", "school": "Columbia",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Kristina Megerdichian", "school": "Tufts",   "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Kuangye Wang", "school": "Columbia",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Kyle Coopersmith", "school": "University of Pittsburgh",   "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Kyle Quinlan", "school": "Yale",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Kylie Feliciano", "school": "Boston University",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Kyra Haddad", "school": "Brown",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Lamisa Khan", "school": "NYU",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Latifa Fasla", "school": "Brandeis",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Laura Howard", "school": "University of Virginia",   "initial_year": 2021, "last_year_seen": 2021},
    {"debater": "Laura Zhang", "school": "Princeton",   "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Lauren Fanter", "school": "Temple",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Lauren Kim", "school": "Wellesley",   "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Lavina Ngo", "school": "Smith",   "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Leander Quiroz", "school": "Wellesley",   "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Leemah Bisht", "school": "Maryland",   "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Leia Park", "school": "Carnegie Mellon",   "initial_year": 2022, "last_year_seen": 2022},   {"debater": "Leigh Murphy", "school": "Williams",   "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Juan Diego Cisneros", "school": "Lehigh",   "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Pragna Yalamanchili", "school": "Maryland", "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Bao Nghi Ho", "school": "Maryland", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Aaliyah Bullen", "school": "Swarthmore", "initial_year": 2022, "last_year_seen": 2022},   {"debater": "Aaradhya Diwan", "school": "Harvard", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Kiran Subramanian", "school": "Rutgers", "initial_year": 2021, "last_year_seen": 2024}, {"debater": "Frank Rodriguez", "school": "Northeastern", "initial_year": 2019, "last_year_seen": 2023}, {"debater": "Oden", "school": "NYU", "initial_year": 2025, "last_year_seen": 2025},
    {"debater": "Leon Gold", "school": "University of Chicago", "initial_year": 2022, "last_year_seen": 2024}, {"debater": "Abby Sweeney", "school": "George Washington", "initial_year": 2020, "last_year_seen": 2022}, {"debater": "Chris Vanderpool", "school": "Brown", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Zora Kuehne", "school": "Haverford", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Yese Erazo-Tequianes", "school": "Pace", "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Aatikah Awan", "school": "NYU", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Abby Garber", "school": "Wellesley", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Abby Morin", "school": "American", "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Abhinav Agarwal", "school": "Princeton", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Abhinav Aitha", "school": "Fordham", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Abigail Romero", "school": "Harvard", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Abrar Ahmed", "school": "Columbia", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Adam Gould", "school": "Brandeis", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Adam Rusakow", "school": "University of Chicago", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Adi Chattopadhyay", "school": "Swarthmore", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Adrian Seferlis", "school": "Rutgers", "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Adrien Amouroux", "school": "University of Chicago", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Advesh Jalan", "school": "Yale", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Ahan Raina", "school": "University of Chicago", "initial_year": 2021, "last_year_seen": 2022}, {"debater": "Aidan Liu", "school": "University of Chicago", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Aishwarya Rajapur", "school": "Amherst", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Aj Matos", "school": "Bates", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Akber Latif", "school": "George Washington", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Alaska Irr", "school": "Brandeis", "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Albert Hao", "school": "Columbia", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Aleeza Syed", "school": "Northeastern", "initial_year": 2023, "last_year_seen": 2025},  {"debater": "Paul Marcelli", "school": "Johns Hopkins",   "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Juan Diego Cisneros", "school": "Lehigh", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Pragna Yalamanchili", "school": "Maryland", "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Bao Nghi Ho", "school": "Maryland", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Alessandra Lorenzo", "school": "University of Chicago", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Alex Bennett", "school": "Brandeis", "initial_year": 2020, "last_year_seen": 2022},
    {"debater": "Alex Rizzo", "school": "American", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Alex Shieh", "school": "Brown", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Alexa Sanchez", "school": "Smith", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Alexander Mclaren", "school": "Columbia", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Alexander West", "school": "Temple", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Alexander Yankovsky", "school": "Fordham", "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Alexis Walker", "school": "University of Virginia", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Alia Bonanno", "school": "Columbia", "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Alia Derriey", "school": "Binghamton University", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Alia Kafil", "school": "Columbia", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Alice Yang", "school": "Drexel", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Alina Antropova", "school": "University of Massachusetts", "initial_year": 2023, "last_year_seen": 2024},   {"debater": "Alina Palacios", "school": "Swarthmore", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Allie Rosenstein", "school": "Harvard", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Almila Muslu", "school": "Drexel", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Amadou Thiam", "school": "Prince George's Community College", "initial_year": 2025, "last_year_seen": 2025},
    {"debater": "Amanda Blatz", "school": "Haverford", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Amarynth Ruch", "school": "Temple", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Amber Gao", "school": "NYU", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Ambika Kandasamy", "school": "Johns Hopkins", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Ameena Ahmed", "school": "University of Chicago", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Ameia Booker", "school": "University of Chicago", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Amelia Le", "school": "Lehigh", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Anantha Kashibhatla", "school": "Rutgers", "initial_year": 2019, "last_year_seen": 2020},
    {"debater": "Anastasia Malenko", "school": "Stanford", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Andrea Zhou", "school": "University of Chicago", "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Andrew Cole", "school": "University of Pittsburgh", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Andrew Conkey", "school": "University of Chicago", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Andrew Hoffman", "school": "William and Mary", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Andrew Vandenbussche", "school": "Penn", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Angel Paz", "school": "NYU", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Anirudh Narsipur", "school": "Brown", "initial_year": 2021, "last_year_seen": 2022},
    {"debater": "Anisah Colón", "school": "Villanova", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Anish Kanthameni", "school": "NYU", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Anjali Agarwal", "school": "Haverford", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Anna Ast", "school": "Boston University", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Anna Krans", "school": "Yale", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Anna Nowalk", "school": "Fordham", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Alison Linares", "school": "Yale", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Annabella Campbell", "school": "Fordham", "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Alekhya Bhat", "school": "Wellesley", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Annie Chi", "school": "Princeton", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Annie Dong", "school": "Penn", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Annie Zhi", "school": "University of Chicago", "initial_year": 2020, "last_year_seen": 2022},  {"debater": "Anouk Yeh", "school": "Yale", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Anshika Agrawal", "school": "Johns Hopkins", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Anthony Boss", "school": "Brown", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Anuka Upadhye", "school": "George Washington", "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Anya Rohatgi", "school": "Fordham", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Aogay Alozai Wardak", "school": "Tufts", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Apollo Grimes", "school": "Brandeis", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Ari Gershengorn", "school": "NYU", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Ariane Sharifi", "school": "Maryland", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Anya Rohtagi", "school": "Wellesley", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Arjun Suryawanshi", "school": "Penn", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Armaan Sheth", "school": "University of Massachusetts", "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Aron Ravin", "school": "Yale", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Arthur Yolles", "school": "George Washington", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Arushi Kaushik", "school": "NYU", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Arushi Sahay", "school": "NYU", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Arya Nalluri", "school": "American", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Asher Mendelson", "school": "NYU", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Ashkay Srivasan", "school": "Columbia", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Ashley Huang", "school": "Tufts", "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Athena Atsides", "school": "George Washington", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Audri Bhomick", "school": "Brandeis", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Austin Chapman", "school": "Maryland", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Austin Chen", "school": "Brandeis", "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Austin Riegel", "school": "Rutgers", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Austin Zheng", "school": "Bowdoin College", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Autri Basu", "school": "Amherst", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Ava Desantis", "school": "George Washington", "initial_year": 2019, "last_year_seen": 2020},
    {"debater": "Ava Nagy", "school": "Boston University", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Ava Rahman", "school": "Brown", "initial_year": 2024, "last_year_seen": 2024},  {"debater": "Avery Lenihan", "school": "Yale", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Avery Li", "school": "William and Mary", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Awsam Bouabid", "school": "Northeastern", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Ayaan De Silva", "school": "University of Chicago", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Ayva Kacir", "school": "Tufts", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Bairavi Sundaram", "school": "George Washington", "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Barna Marczali", "school": "Johns Hopkins", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Becca Carer", "school": "Johns Hopkins", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Ben Bradley", "school": "Brown", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Ben Fica", "school": "University of Chicago", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Ben Fitzgerald", "school": "Haverford", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Ben Hinish", "school": "Drexel", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Ben Skarbek", "school": "Temple", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Ben Wilson", "school": "Maryland", "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Ben Wong", "school": "Binghamton University", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Benjamin Xu", "school": "Williams", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Berk Turkkani", "school": "NYU", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Betty Qian", "school": "Wellesley", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Bey Soriano", "school": "Johns Hopkins", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Bhavya Surapaneni", "school": "Penn", "initial_year": 2023, "last_year_seen": 2024},     {"debater": "Bianca Ferreira", "school": "Temple", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Bilal Tariq", "school": "Amherst", "initial_year": 2023, "last_year_seen": 2024},
    {"debater": "Bill Chen", "school": "Harvard", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Blair Peng", "school": "University of Chicago", "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Bradley Evans", "school": "University of Pittsburgh", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Brandon Bosaz", "school": "Maryland", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Breanna Crossman", "school": "George Washington", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Brendan Brestage", "school": "Northeastern", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Brennan Mcdermott", "school": "American", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Brian O'Neill", "school": "Tufts", "initial_year": 2021, "last_year_seen": 2021},
    {"debater": "Brittany Bin", "school": "MIT", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Bryan Mcdonough", "school": "University of Massachusetts", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Bryan Thomas", "school": "Northeastern", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Bryce Trent", "school": "Williams", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Cayden Monteiro", "school": "George Washington", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Carley Calfee", "school": "American", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Brynn Kroke", "school": "Brown", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Chansol Park", "school": "NYU", "initial_year": 2025, "last_year_seen": 2025}, 
    {"debater": "Daisy Bateman", "school": "American", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Fabiha Era", "school": "Binghamton University", "initial_year": 2024, "last_year_seen": 2025}, {"debater": "George Anderson", "school": "William and Mary", "initial_year": 2019, "last_year_seen": 2021}, {"debater": "Hassan Looky", "school": "Harvard", "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Zachary Braunstein", "school": "Maryland", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Tyler Hugo", "school": "Temple", "initial_year": 2022, "last_year_seen": 2024}, {"debater": "Tess Yu", "school": "Johns Hopkins", "initial_year": 2021, "last_year_seen": 2022}, {"debater": "Akhil Mallajosyula", "school": "Maryland", "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Callum Mcfarlane", "school": "University of Chicago", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Cameron Sagheb", "school": "Georgetown", "initial_year": 2021, "last_year_seen": 2022}, {"debater": "Camille Jones", "school": "Princeton", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Chloe Yu", "school": "Maryland", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Connor Beaney", "school": "Brandeis", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Ellie Papraniku", "school": "CUNY", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Evan Tao", "school": "Brown", "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Zoe Rose", "school": "William and Mary", "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Zoe Raptis", "school": "Tufts", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Yosua Siagian", "school": "George Washington", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Yasmine Dweir", "school": "NYU", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Varun Singh", "school": "Maryland", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Wayland Bardwell", "school": "Boston University", "initial_year": 2022, "last_year_seen": 2024}, {"debater": "Trevor Kickliter", "school": "University of Pittsburgh", "initial_year": 2019, "last_year_seen": 2020}, {"debater": "Davin Bhatti", "school": "University of Chicago", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Clarity Houts", "school": "Amherst", "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Cleo Elrashidy", "school": "Brown", "initial_year": 2019, "last_year_seen": 2020}, {"debater": "Catherine Horner", "school": "Dartmouth", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Carina Olivar", "school": "Fordham", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Carme Sanz-Muñoz", "school": "Wellesley", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Caroline Hennigan", "school": "Harvard", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Caroline Loveday", "school": "William and Mary", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Caroline Sagristino", "school": "George Washington", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Claire Fennell", "school": "NYU", "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Sriya Uttharkar", "school": "Rutgers", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Sophia Mason", "school": "Pace", "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Smiti Modhurima", "school": "Columbia", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Zoey Morris", "school": "Stanford", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Zakiriya Gladney", "school": "Harvard", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Yvette Shu", "school": "Boston University", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Yuri Izumikawa", "school": "Johns Hopkins", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Yuqian Li", "school": "Boston University", "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Yuntian Gan", "school": "Brandeis", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Youssef Bousada", "school": "NYU", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Yonah Gross", "school": "Boston University", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Yoav Rafalin", "school": "Carnegie Mellon", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Yinjun Chen", "school": "Boston University", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Yilang Fan", "school": "NYU", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Yijiao Guo", "school": "Yale", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Yifei Sun", "school": "Bentley University", "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Yazeed Abayazid", "school": "University of Chicago", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Yaxuan Li", "school": "Carnegie Mellon", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Yasmin Roach", "school": "Yale", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Yaroslav Opanasyuk", "school": "CUNY", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Yannik Omictin", "school": "George Washington", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Yannick Daoud", "school": "NYU", "initial_year": 2020, "last_year_seen": 2020},  {"debater": "Yanni Trimikliniotis", "school": "NYU", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Yan Ning", "school": "Brown", "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Xinyue Gu", "school": "Johns Hopkins", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Wyatt Smith", "school": "Williams", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Winnie Jiang", "school": "Yale", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Willow West", "school": "Fordham", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Willie Gomez", "school": "University of Massachusetts", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "William Mcadams", "school": "Washington University in St. Louis", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Will Florentino", "school": "Georgetown", "initial_year": 2021, "last_year_seen": 2021},
    {"debater": "Wasi Ahmed", "school": "CUNY", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Wasan Rafat", "school": "Harvard", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Walker Evans", "school": "American", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Vinya Lingamneni", "school": "Brandeis", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Vincent Shen", "school": "Rutgers", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Vik Li", "school": "Northeastern", "initial_year": 2019, "last_year_seen": 2019},     {"debater": "Vaughn Battista", "school": "Rutgers", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Vara Qi Gunananthan", "school": "Johns Hopkins", "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Udi Akolkar", "school": "Carnegie Mellon", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Tyler Klinedinst", "school": "American", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Tuong Do", "school": "Haverford", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Triet Le", "school": "Fordham", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Trey Garcia-Schartz", "school": "Fordham", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Trevor Haefner", "school": "George Washington", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Ting Hsu", "school": "Boston University", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Tina Tang", "school": "Georgetown", "initial_year": 2024, "last_year_seen": 2025},
    {"debater": "Tim Brennan", "school": "Binghamton University", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Thulasi Varatharajan", "school": "University of Pittsburgh", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Thomas Lin", "school": "University of Chicago", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Theo Lockrow", "school": "Wesleyan", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Thabang Matona", "school": "Brandeis", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Tendai Coady", "school": "Williams", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Tanaz Bari", "school": "Binghamton University", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Tamaki Sugihara", "school": "University of Massachusetts", "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Talia Katz", "school": "University of Chicago", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Tabitha Chua", "school": "Boston University", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Renny Jiang", "school": "Brown", "initial_year": 2021, "last_year_seen": 2023}, {"debater": "Riker Wachtler", "school": "Boston University", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Peter Capote", "school": "Rutgers", "initial_year": 2024, "last_year_seen": 2024},	 {"debater": "Omar Elalaoui", "school": "Rutgers", "initial_year": 2023, "last_year_seen": 2024},	 {"debater": "Peter Heller", "school": "William and Mary", "initial_year": 2019, "last_year_seen": 2021}, {"debater": "Penelope Toll", "school": "University of Chicago", "initial_year": 2022, "last_year_seen": 2022},	
    {"debater": "Nyankpani Kesson Abdul-Quddud", "school": "Tufts", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Natalia Caid", "school": "Temple", "initial_year": 2024, "last_year_seen": 2026}, {"debater": "Nafiz Zaman", "school": "Johns Hopkins", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Mary Clarke", "school": "Brown", "initial_year": 2022, "last_year_seen": 2025}, {"debater": "Mateo Mcnamara", "school": "George Washington", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Mandy Feuerman", "school": "Brandeis", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Madigan Webb", "school": "William and Mary", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Sophie Mason", "school": "Pace", "initial_year": 2024, "last_year_seen": 2025},
    {"debater": "Stanley Sun", "school": "Northeastern", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Stephanie Laplante", "school": "University of Massachusetts", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Stephen Scopa", "school": "Brown", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Saxon Hart", "school": "William and Mary", "initial_year": 2019, "last_year_seen": 2022}, {"debater": "Sanket Bhalotia", "school": "Fordham", "initial_year": 2023, "last_year_seen": 2025}, {"debater": "Sarah Barkatz", "school": "CUNY", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Vishakh Sandwar", "school": "NYU", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Travis Hunsberger", "school": "University of Pittsburgh", "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Taylor Small", "school": "Brandeis", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Talia Yett", "school": "Brown", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Sándor Lorange", "school": "Tufts", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Supriyaa Hejib", "school": "University of Massachusetts", "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Sunint Bindra", "school": "Dartmouth", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Summit Sarkar", "school": "Amherst", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Sudhish Rao", "school": "Johns Hopkins", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Steven Macawili", "school": "Boston University", "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Steven Lou", "school": "Princeton", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Stacy Amoako", "school": "University of Massachusetts", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Srivatsav Pyda", "school": "Columbia", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Sree Dharmaraj", "school": "Brandeis", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Sravya Dontharaju", "school": "Tufts", "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Sophie Fetter", "school": "Villanova", "initial_year": 2021, "last_year_seen": 2024}, {"debater": "Sophia Marmai", "school": "University of Massachusetts", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Sophia Farinella", "school": "University of Massachusetts", "initial_year": 2022, "last_year_seen": 2024},
    {"debater": "Sophia Anderson", "school": "University of Delaware", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Sonam Tenzin", "school": "Yale", "initial_year": 2021, "last_year_seen": 2023}, {"debater": "Sofia Little", "school": "Rutgers", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Skyler Goldberg", "school": "Tufts", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Skylar Jones", "school": "NYU", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Simran Singh", "school": "Brown", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Sierra Maciorowski", "school": "Stanford", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Siddhant Moily", "school": "Brandeis", "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Sidd Jain", "school": "University of Chicago", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Sid Gupta", "school": "Maryland", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Sia Kothari", "school": "Bentley University", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Shriya Sane", "school": "Penn", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Skyler Lee", "school": "Amherst", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Shriya Kosuru", "school": "William and Mary", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Shreyam Misra", "school": "University of Chicago", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Shlomo Grun", "school": "CUNY", "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Shiqi Peng", "school": "Brandeis", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Sheoli Lele", "school": "William and Mary", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Shayna Leng", "school": "Harvard", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Shanyu Thibaut Juneja", "school": "University of Massachusetts", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Seth Choi", "school": "Johns Hopkins", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Serena Salam", "school": "Wellesley", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Senthil", "school": "Boston University", "initial_year": 2022, "last_year_seen": 2022},{"debater": "Sefat Samee", "school": "Odette", "initial_year": 2020, "last_year_seen": 2020},
    {"debater": "Sebastien Ludwig", "school": "University of Chicago", "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Sebastian Pollock", "school": "Amherst", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Sean Roth", "school": "University of Massachusetts", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Sean Oh", "school": "Penn", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Sean Nolan", "school": "Northeastern", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Sean Berman", "school": "Brandeis", "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Scarlett Wang", "school": "Bates", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Sasya Koneru", "school": "Fordham", "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Sasquatch Ray", "school": "Harvard", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Sarwa Shah", "school": "Wellesley", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Sare King", "school": "Brandeis", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Sarah Hickey", "school": "Wellesley", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Samyak Jain", "school": "George Washington", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Samuel Rohwer", "school": "Rutgers", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Samuel Butler", "school": "University of Chicago", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Samra Lulseged", "school": "Penn", "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Samarth Jha",  "school": "Bates", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Sam Slack", "school": "Bentley University", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Sam Siemer", "school": "Johns Hopkins", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Sam Sagawa", "school": "University of Chicago", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Sam Russo", "school": "Tufts", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Sam Rohwer", "school": "Rutgers", "initial_year": 2019, "last_year_seen": 2022}, {"debater": "Sam Passner", "school": "University of Virginia", "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Sam Mackin", "school": "University of Massachusetts", "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Sajid Ibrahim", "school": "Boston University", "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Sabrina Yang", "school": "NYU", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Sabrina Eager", "school": "William and Mary", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Ryan Yang", "school": "MIT", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Ryan Flammer", "school": "University of Virginia", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Ryan Craig", "school": "William and Mary", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Rushabh Patel", "school": "George Washington", "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Rui Gong", "school": "Boston University", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Ross Khelemsky", "school": "Penn", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Roshan Pillai", "school": "University of Massachusetts", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Roli Tinsley", "school": "American", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Rohita Krishnakumar", "school": "Maryland", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Rizky Ananda", "school": "Penn", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Riya Bhattacharjee", "school": "Wellesley", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Rishi Mukherjee", "school": "University of Massachusetts", "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Rija Masroor", "school": "William and Mary", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Richie Lu", "school": "Columbia", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Richard Kim", "school": "Yale", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Renee Wu", "school": "Johns Hopkins", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Rene Garrett", "school": "Denison", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Rebecca Mollet", "school": "Maryland", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Rebecca Hsu", "school": "University of Pittsburgh", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Rebeca Samano", "school": "American", "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Raymond Banke", "school": "Columbia", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Ray Yang", "school": "Boston University", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Rana Ürek", "school": "Columbia", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Ramisa Rahman", "school": "William and Mary", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Rainiero Disera", "school": "Brandeis", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Ragy Amin", "school": "University of Chicago", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Raghad Mohamed", "school": "Bowdoin College", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Rafi Chowdhury", "school": "William and Mary", "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Rafay Abdul", "school": "Bates", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Rachel Liu", "school": "Boston University", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Qingxi Jaja Wang", "school": "Johns Hopkins", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Pulin Wang", "school": "Maryland", "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Priyanka Mahat", "school": "Brown", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Presley Forrest", "school": "University of Massachusetts", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Pre Ferri", "school": "Temple", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Pratham Lakhani", "school": "Carnegie Mellon", "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Prakhar Agrawal", "school": "Amherst", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Prajata Roy", "school": "NYU", "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Pragnya Yerramalli", "school": "Lehigh", "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Peter Mao", "school": "Johns Hopkins", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Peter Lawrence", "school": "Maryland", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Pesandi Gunasekera", "school": "University of Virginia", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Patrick Song", "school": "Rutgers", "initial_year": 2019, "last_year_seen": 2020}, {"debater": "Paneez Oilai", "school": "Georgetown", "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Owen Boice", "school": "American", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Ovia Sundar", "school": "Tufts", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Oscar Barrios", "school": "Princeton", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Omogolo Pikinini", "school": "Lehigh", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Omar Khan", "school": "University of Chicago", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Olivia Taboada", "school": "Temple", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Olivia Parashar", "school": "Brandeis", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Olivia Mclane", "school": "Temple", "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Olivia Mastrangelo", "school": "William and Mary", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Olivia Martinez", "school": "Smith", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Olivia Lowry", "school": "Johns Hopkins", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Olivia Ferrier", "school": "University of Delaware", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Oliver Brazda", "school": "Tufts", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Zach Fedyk", "school": "University of Pittsburgh", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Winn Ryan", "school": "Johns Hopkins", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Tyler Viljaste", "school": "University of Pittsburgh", "initial_year": 2020, "last_year_seen": 2020},
    {"debater": "Topher Zane", "school": "William and Mary", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Shravani Subhedar", "school": "Fordham", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Sebastian Rajguru", "school": "William and Mary", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Ryan Shue", "school": "William and Mary", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Ralph Montas Osias", "school": "William and Mary", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Pierre Mathier", "school": "Wesleyan", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Nusayba Chowdhury", "school": "University of Pittsburgh", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Nora Sam", "school": "Rutgers", "initial_year": 2019, "last_year_seen": 2019},
    {"debater": "Noor Al-Saloum",  "school": "Johns Hopkins", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Noe Cifuentes", "school": "Drexel", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Noah Ogata", "school": "George Washington", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Noah Berman", "school": "Northeastern", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Nitin Kumar", "school": "Northeastern", "initial_year": 2021, "last_year_seen": 2023}, {"debater": "Nikola Simon", "school": "University of Chicago", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Nikita Rodin", "school": "University of Chicago", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Nikhil Rao", "school": "William and Mary", "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Nico Llorente Valin", "school": "Fordham", "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Nick Freilino", "school": "Duquesne University", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Nick Claudio", "school": "George Washington", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Nick Bukofsky", "school": "Binghamton University", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Nicholas Sanchez", "school": "Fordham", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Nicholas Perez", "school": "Johns Hopkins", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Nicholas Kelly", "school": "Harvard", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Nicholas Hao", "school": "University of Chicago", "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Nicholas Chuckas", "school": "University of Virginia", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Nic Howayeck", "school": "University of Massachusetts", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Nehemiah Cesar", "school": "Williams", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Natt Sears", "school": "Georgetown", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Nathaniel Maali-Pohl", "school": "University of Massachusetts", "initial_year": 2026, "last_year_seen": 2026}, {"debater": "Nathan Tang", "school": "Bentley University", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Nathan Schechter", "school": "Haverford", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Nathan Peng", "school": "University of Chicago", "initial_year": 2020, "last_year_seen": 2020},
    {"debater": "Nathan De Moura", "school": "Harvard", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Natasha Raman", "school": "Dartmouth", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Natalie George", "school": "Odette", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Natalia Zorrilla", "school": "Princeton", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Natalia Siwek", "school": "Harvard", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Nadia Lee", "school": "Maryland", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Nadeen Alomar", "school": "Maryland", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Muskan Chhabra", "school": "George Washington", "initial_year": 2022, "last_year_seen": 2022}, 
    {"debater": "Muhammad Siddiqui", "school": "NYU", "initial_year": 2024, "last_year_seen": 2025}, {"debater": "Muhammad Dhafer", "school": "Stanford", "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Mounisha Anumolu", "school": "Dartmouth", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Monica Li", "school": "Rutgers", "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Mohammed Sarker", "school": "Penn", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Mitchell Green", "school": "Wesleyan", "initial_year": 2020, "last_year_seen": 2022}, {"debater": "Mireya Sanchez-Maes", "school": "Harvard", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Mirada Makhmutova", "school": "Boston University", "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Miles Richardson", "school": "George Washington", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Miles Gendebien", "school": "Tufts", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Mido Sang", "school": "University of Chicago", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Michelle Doan", "school": "Amherst", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Michael Yoo", "school": "Johns Hopkins", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Michael Tsutagawa", "school": "Johns Hopkins", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Michael Tatum", "school": "Boston College", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Michael Smith", "school": "University of Delaware", "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Michael Hong", "school": "Northeastern", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Michael Clifford", "school": "University of Massachusetts", "initial_year": 2023, "last_year_seen": 2024}, {"debater": "Michael Chen", "school": "Penn", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Michael Carley", "school": "George Washington", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Michael Avila", "school": "Binghamton University", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Micah Kawecki", "school": "University of Virginia", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Mia Scherer", "school": "Wellesley", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Mia Madonna", "school": "NYU", "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Mia Kuchner", "school": "American", "initial_year": 2019, "last_year_seen": 2021}, {"debater": "Mesoun Hassan", "school": "American", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Mercer Mercer", "school": "Rutgers", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Meghan Hendrix", "school": "University of Chicago", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Megan Williams", "school": "American", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Maxwell Weiner", "school": "Brandeis", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Maxwell Guo", "school": "NYU", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Max Zhou", "school": "Hamilton", "initial_year": 2024, "last_year_seen": 2024}, 
    {"debater": "Matthew Sinning", "school": "Hamilton", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Matthew Li", "school": "William and Mary", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Matt Kiley", "school": "Harvard", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Matilda Stricherz", "school": "Stanford", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Mary Wu", "school": "Boston University", "initial_year": 2022, "last_year_seen": 2023}, {"debater": "Marjola Demollari", "school": "University of Massachusetts", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Marissa Pereira", "school": "University of Delaware", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Marina Pantner", "school": "William and Mary", "initial_year": 2020, "last_year_seen": 2020},
    {"debater": "Marilyn Santo", "school": "Georgetown", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Mariam Nageeb", "school": "Fordham", "initial_year": 2023, "last_year_seen": 2023},   {"debater": "Maria Dubasov", "school": "William and Mary", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Marcelo Rodriguez Parra", "school": "Brown", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Marc Speeches", "school": "Brown", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Marah Sami", "school": "Amherst", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Manhua Kim", "school": "Maryland", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Maksym Sherman", "school": "Swarthmore", "initial_year": 2021, "last_year_seen": 2021},
    {"debater": "Mai Al Shaaban", "school": "Brandeis", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Mahir Abrar", "school": "Lehigh", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Madeline Wyatt", "school": "Columbia", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Madeline Turner", "school": "Northeastern", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Madeleine Eichorn", "school": "George Washington", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Mac O'Hara", "school": "Penn", "initial_year": 2019, "last_year_seen": 2020}, {"debater": "Luke Chan", "school": "Princeton", "initial_year": 2021, "last_year_seen": 2021}, {"debater": "Lukas Roybal", "school": "Columbia", "initial_year": 2024, "last_year_seen": 2024},
    {"debater": "Luka Bulic Braculj", "school": "MIT", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Luis Luna", "school": "Cornell", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Lucia Gonzalez", "school": "Penn", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Lucas Irwin", "school": "Princeton", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Luca Montoya",  "school": "Swarthmore", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Louis Mukama",  "school": "Harvard", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Lorenzo Songsare-Shevy", "school": "Bates", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "London Cooper", "school": "Columbia", "initial_year": 2023, "last_year_seen": 2023},
    {"debater": "Logan Quick", "school": "Harvard", "initial_year": 2020, "last_year_seen": 2022}, {"debater": "Logan De Raspide Ross", "school": "Boston University", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Lizzie Kerman", "school": "William and Mary", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Liz Lin Moore", "school": "Wellesley", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Liz Greer", "school": "George Washington", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Linh Hoang", "school": "Fordham", "initial_year": 2025, "last_year_seen": 2025}, {"debater": "Lindsey White",  "school": "University of North Carolina", "initial_year": 2024, "last_year_seen": 2024}, {"debater": "Lindsey Greenberg", "school": "Northeastern", "initial_year": 2022, "last_year_seen": 2022},
    {"debater": "Lindsay Khalluf", "school": "Georgetown", "initial_year": 2022, "last_year_seen": 2022}, {"debater": "Lily Siru", "school": "Haverford", "initial_year": 2023, "last_year_seen": 2022}, {"debater": "Lily Li", "school": "Haverford", "initial_year": 2020, "last_year_seen": 2020}, {"debater": "Liam Rosario", "school": "Binghamton University", "initial_year": 2019, "last_year_seen": 2019}, {"debater": "Liam Bloom", "school": "Wesleyan", "initial_year": 2023, "last_year_seen": 2023}, {"debater": "Lemuel Yu", "school": "Boston University", "initial_year": 2022, "last_year_seen": 2022}, 
]

recovered_debaters = pd.DataFrame(recovered_debaters)
recovered_debaters = recovered_debaters.rename(columns={"initial_year": "initial_year_seen"})
recovered_debaters = recovered_debaters.rename(columns={"last_year_seen": "final_year_seen"})
recovered_debaters.to_csv("Data_Curation_Data/Recovered_Debaters.csv", index=False)
valid_debaters = pd.read_csv("Data_Curation_Data/Debater_Roster.csv")
valid_debaters = pd.concat([valid_debaters, recovered_debaters], ignore_index=True)
valid_debaters = valid_debaters.drop_duplicates()
valid_debaters.to_csv("Data_Curation_Data/Combined_Debaters.csv", index=False)

Now I need to update my round dataframe so that they contain the newly added judges and debaters. This code utilizes the functions written above.

In [16]:
judge_df_after = add_judges_info("Data_Curation_Data/Combined_Debaters.csv")
judge_df_after.to_csv("Data_Curation_Data/rounds_After_Judge_Restoration.csv", index=False)
placeholder = add_debater_info("Data_Curation_Data/Combined_Debaters.csv", "Data_Curation_Data/rounds_After_Judge_Restoration.csv", "Data_Curation_Data/rounds_After_Debater_Restoration.csv")

I also decided to reverify the years that each debater first and last appears due to the apparent tracking issues in the tab website. If 989 debaters were never registered then it is entirely possible other debaters were not registered in years they did compete in, causing issues in the years tracking I performed. To resolve this, I made the following to update the "Combined_Debaters.csv" Initial year Seen and Final year Seen if differing from their actual appearences in the data.

In [18]:
df_clean_rounds = pd.read_csv("Data_Curation_Data/rounds_After_Debater_Restoration.csv")
df_clean_debaters = pd.read_csv("Data_Curation_Data/Combined_Debaters.csv")

role_pairs = [
    ("speaker_one_name", "speaker_one_school"),
    ("speaker_two_name", "speaker_two_school"),
    ("opponent_one_name", "opponent_one_school"),
    ("opponent_two_name", "opponent_two_school"),
    ("judge", "judge_school")
]

long_df = pd.concat([
    df_clean_rounds[[name_col, school_col, "season", "year"]]
        .rename(columns={name_col: "debater", school_col: "school"})
    for name_col, school_col in role_pairs
], ignore_index=True).dropna(subset=["debater", "school"])

season_order = {"Fall": 0, "Spring": 1}
long_df["_season_key"] = long_df["year"] * 10 + long_df["season"].map(season_order)

# Convert calendar year -> season-label year (the "2004" in "2004-05")
# Fall keeps its year; Spring belongs to the season that started the PREVIOUS calendar year
season_offset = {"Fall": 0, "Spring": 1}
long_df["season_start_year"] = long_df["year"] - long_df["season"].map(season_offset)

idx_first = long_df.groupby(["debater", "school"])["_season_key"].idxmin()
idx_last = long_df.groupby(["debater", "school"])["_season_key"].idxmax()

first_seen = long_df.loc[idx_first, ["debater", "school", "season_start_year"]].rename(
    columns={"season_start_year": "computed_initial_year"}
)
last_seen = long_df.loc[idx_last, ["debater", "school", "season_start_year"]].rename(
    columns={"season_start_year": "computed_final_year"}
)

first_last = pd.merge(first_seen, last_seen, on=["debater", "school"])

df_clean_debaters = pd.merge(df_clean_debaters, first_last, on=["debater", "school"], how="left")

df_clean_debaters["initial_year_seen"] = df_clean_debaters["initial_year_seen"].astype("Int64")
df_clean_debaters["final_year_seen"] = df_clean_debaters["final_year_seen"].astype("Int64")
df_clean_debaters["computed_initial_year"] = df_clean_debaters["computed_initial_year"].astype("Int64")
df_clean_debaters["computed_final_year"] = df_clean_debaters["computed_final_year"].astype("Int64")

df_clean_debaters["initial_year_corrected"] = (
    df_clean_debaters["computed_initial_year"].notna() &
    (df_clean_debaters["computed_initial_year"] != df_clean_debaters["initial_year_seen"])
)
df_clean_debaters["final_year_corrected"] = (
    df_clean_debaters["computed_final_year"].notna() &
    (df_clean_debaters["computed_final_year"] != df_clean_debaters["final_year_seen"])
)

df_clean_debaters["initial_year_seen"] = df_clean_debaters["initial_year_seen"].where(
    ~df_clean_debaters["initial_year_corrected"], df_clean_debaters["computed_initial_year"]
)
df_clean_debaters["final_year_seen"] = df_clean_debaters["final_year_seen"].where(
    ~df_clean_debaters["final_year_corrected"], df_clean_debaters["computed_final_year"]
)

df_clean_debaters = df_clean_debaters.drop(
    columns=["computed_initial_year", "computed_final_year",
             "initial_year_corrected", "final_year_corrected"]
)

df_clean_debaters.to_csv("Data_Curation_Data/Combined_Debaters.csv", index=False)

One last pass to update the years columns.

In [19]:
judge_df_after = add_judges_info("Data_Curation_Data/Combined_Debaters.csv")
judge_df_after.to_csv("Data_Curation_Data/rounds_Final_Judge_Restoration.csv", index=False)
placeholder = add_debater_info("Data_Curation_Data/Combined_Debaters.csv", "Data_Curation_Data/rounds_Final_Judge_Restoration.csv", "Data_Curation_Data/rounds_Final_Debater_Restoration.csv")

Reorder for clarity.

In [20]:
df_clean_rounds = pd.read_csv("Data_Curation_Data/rounds_Final_Debater_Restoration.csv")
df_clean_rounds['tournament_key'] = (
    df_clean_rounds['tournament'].astype(str).str.strip() + ' ' +
    df_clean_rounds['season'].astype(str).str.strip() + ' ' +
    df_clean_rounds['year'].astype(str).str.strip()
)

new_order = ['tournament_key', 'year', 'season', 'tournament', 'round', 'g_o', 'w_l', 
             'speaker_one_name', 'speaker_one_status', 'speaker_one_speaks', 'speaker_one_rank', 'speaker_one_school', 'speaker_one_initial_year', 'speaker_one_last_competed',
             'speaker_two_name',  'speaker_two_status', 'speaker_two_speaks', 'speaker_two_rank', 'speaker_two_school', 'speaker_two_initial_year', 'speaker_two_last_competed',
             'opponent_one_name',  'opponent_one_status', 'opponent_one_speaks', 'opponent_one_rank', 'opponent_one_school', 'opponent_one_initial_year', 'opponent_one_last_competed',
             'opponent_two_name',  'opponent_two_status', 'opponent_two_speaks', 'opponent_two_rank', 'opponent_two_school', 'opponent_two_initial_year', 'opponent_two_last_competed',
             'judge', 'judge_school', 'judge_initial_year', 'judge_last_competed']
df_clean_rounds = df_clean_rounds.reindex(columns=new_order)
df_clean_rounds.to_csv("Data_Curation_Data/rounds_Final_Debater_Restoration.csv", index=False)

Create column containing Judge Experience and separate out the sides and wins from the w_l and g_o columns.

In [23]:
df_all = pd.read_csv("Data_Curation_Data/rounds_Final_Debater_Restoration.csv")
df_roster_all = pd.read_csv("Data_Curation_Data/Combined_Debaters.csv")

# Assuming roster has Name/Debater and Initial year Seen columns — adjust to your actual column names
roster_lookup = df_roster_all.drop_duplicates(subset=["debater"]).set_index("debater")["initial_year_seen"]

df_all["judge_initial_year"] = df_all["judge"].map(roster_lookup)

df_all["judge_years_experience"] = df_all["year"] - df_all["judge_initial_year"]

# No match in roster -> treat as unidentified/inexperienced
no_match_mask = df_all["judge_initial_year"].isna()
df_all.loc[no_match_mask, "judge"] = "Unidentified"
df_all.loc[no_match_mask, "judge_years_experience"] = 0

df_all["judge_years_experience"] = df_all["judge_years_experience"].clip(lower=0)

df_all["speaker_side"] = df_all["g_o"]
df_all["opponent_side"] = np.where(df_all["g_o"] == "G", "O", "G")
df_all["speaker_win_status"] = df_all["w_l"]
df_all["opponent_win_status"] = np.where(df_all["w_l"] == "W", "L", "W")
new_order = ['tournament_key', 'year', 'season', 'tournament', 'round', "speaker_side", "opponent_side", "speaker_win_status", "opponent_win_status",
             'speaker_one_name', 'speaker_one_status', 'speaker_one_speaks', 'speaker_one_rank', 'speaker_one_school', 'speaker_one_initial_year', 'speaker_one_last_competed',
             'speaker_two_name',  'speaker_two_status', 'speaker_two_speaks', 'speaker_two_rank', 'speaker_two_school', 'speaker_two_initial_year', 'speaker_two_last_competed',
             'opponent_one_name',  'opponent_one_status', 'opponent_one_speaks', 'opponent_one_rank', 'opponent_one_school', 'opponent_one_initial_year', 'opponent_one_last_competed',
             'opponent_two_name',  'opponent_two_status', 'opponent_two_speaks', 'opponent_two_rank', 'opponent_two_school', 'opponent_two_initial_year', 'opponent_two_last_competed',
             'judge', 'judge_school', 'judge_initial_year', 'judge_last_competed']
df_all = df_all.reindex(columns=new_order)
df_all.insert(0, "id", range(1, len(df_all) + 1))

df_all.to_csv("Data_Curation_Data/Organized_Rounds.csv", index=False)

In [24]:
df_all.isna().sum()

id                               0
tournament_key                   0
year                             0
season                           0
tournament                       0
round                            0
speaker_side                     0
opponent_side                    0
speaker_win_status               0
opponent_win_status              0
speaker_one_name                 0
speaker_one_status               0
speaker_one_speaks               0
speaker_one_rank                 0
speaker_one_school             507
speaker_one_initial_year       507
speaker_one_last_competed      507
speaker_two_name                 0
speaker_two_status               0
speaker_two_speaks               0
speaker_two_rank                 0
speaker_two_school             517
speaker_two_initial_year       517
speaker_two_last_competed      517
opponent_one_name                0
opponent_one_status              0
opponent_one_speaks              0
opponent_one_rank                0
opponent_one_school 

From the above we can see that there is missing information about 2,167 judges, 507 Speaker Ones, 517 Speaker Twos, 1,359 Opponent Ones, and 1389 Opponent Twos. This is acceptable and with no further way of recovering this data, my dataset is now ready for anaylsis.

However, in additon to the rounds and debater data, I will collect further data on the schools and tournaments that occur in APDA in the next sections. The first is a tables of school information that I built from reasearch. For each school found with rosters in the APDA webstie, I found the private/public distinction, the region, and recorded its status as an ivy

In [153]:
%%file Data_Curation_Data/Schools.csv
school,public_private,region,ivy
American,Private,South,Not Ivy
Adelphi,Private,North,Not Ivy
Amherst,Private,North,Not Ivy
Bard,Private,North,Not Ivy
Bates,Private,North,Not Ivy
Bentley University,Private,North,Not Ivy
Berkeley,Public,Central,Not Ivy
Binghamton University,Public,North,Not Ivy
Boston College,Private,North,Not Ivy
Boston University,Private,North,Not Ivy
Bowdoin College,Private,North,Not Ivy
Brierley Price Prior,International,Central,Not Ivy
Bradley,Private,Central,Not Ivy
Brandeis,Private,North,Not Ivy
Brown,Private,North,Ivy
Bryn Mawr,Private,Central,Not Ivy
Bucknell,Private,Central,Not Ivy
Cambridge,International,Central,Not Ivy
Carleton,Private,Central,Not Ivy
Carnegie Mellon,Private,Central,Not Ivy
City College of San Francisco,Public,Central,Not Ivy
Claremont,Private,Central,Not Ivy
Colgate,Private,North,Not Ivy
Columbia,Private,North,Ivy
Columbia Law,Private,North,Ivy
Cornell,Private,North,Ivy
CUNY,Public,Central,Not Ivy
Dalhousie,International,Central,Not Ivy
Dartmouth,Private,North,Ivy
Davidson,Private,South,Not Ivy
Denison,Private,Central,Not Ivy
Drexel,Private,Central,Not Ivy
Duke,Private,South,Not Ivy
Duquesne University,Private,Central,Not Ivy
Durham,International,Central,Not Ivy
Emory,Private,South,Not Ivy
Fairfield,Private,North,Not Ivy
Fisher College,Private,North,Not Ivy
Florida International University,Public,South,Not Ivy
Florida State University,Public,South,Not Ivy
Fordham,Private,North,Not Ivy
FranklinandMarshall,Private,Central,Not Ivy
George Mason,Public,South,Not Ivy
Georgetown,Private,South,Not Ivy
George Washington,Private,South,Not Ivy
Glasgow,International,Central,Not Ivy
Grinnell,Private,Central,Not Ivy
Grove City College,Private,Central,Not Ivy
Hamilton,Private,North,Not Ivy
Hart House,International,Central,Not Ivy
Harvard,Private,North,Ivy
Harvard Law,Private,North,Ivy
Haverford,Private,Central,Not Ivy
Hobart and William Smith,Private,North,Not Ivy
University of Dhaka,International,Central,Not Ivy
IIUM,International,Central,Not Ivy
Johns Hopkins,Private,South,Not Ivy
Kings,International,Central,Not Ivy
Kwame Nkrumah University of Science and Technology,International,Central,Not Ivy
La Verne,Private,Central,Not Ivy
Lehigh,Private,Central,Not Ivy
Loyola Marymount,Private,Central,Not Ivy
Loyola University Chicago,Private,Central,Not Ivy
Maryland,Public,South,Not Ivy
McGill,International,Central,Not Ivy
Middlebury,Private,North,Not Ivy
MIT,Private,North,Not Ivy
Moody Bible Institute,Private,Central,Not Ivy
Morehouse College,Private,South,Not Ivy
Mount Holyoke,Private,North,Not Ivy
Northeastern,Private,North,Not Ivy
Northwestern,Private,Central,Not Ivy
Notre Dame,Private,Central,Not Ivy
NYU,Private,North,Not Ivy
Odette,International,Central,Not Ivy
Ottawa,International,Central,Not Ivy
Oxford,International,Central,Not Ivy
Pace,Private,North,Not Ivy
Patrick Henry,Private,South,Not Ivy
Penn,Private,Central,Ivy
Prince George's Community College,Public,South,Not Ivy
Princeton,Private,Central,Ivy
Providence College,Private,North,Not Ivy
Quakers,Unclear,Central,Not Ivy
Queen's University,International,Central,Not Ivy
RIT,Private,North,Not Ivy
Rochester,Private,North,Not Ivy
RPI,Private,North,Not Ivy
Rutgers,Public,Central,Not Ivy
San Jose State,Public,Central,Not Ivy
Santa Clara,Private,Central,Not Ivy
Simon Fraser University,International,Central,Not Ivy
Simon's Rock College,Private,North,Not Ivy
Skidmore,Private,North,Not Ivy
Smith,Private,North,Not Ivy
Spelman,Private,South,Not Ivy
St. Andrews,International,Central,Not Ivy
Stanford,Private,Central,Not Ivy
St. Johns,Private,North,Not Ivy
St. Mary's,Private,South,Not Ivy
Stony Brook University,Public,North,Not Ivy
Swarthmore,Private,Central,Not Ivy
Syracuse,Private,North,Not Ivy
Tal Aviv,International,Central,Not Ivy
Temple,Public,Central,Not Ivy
TESU,Public,Central,Not Ivy
The College of New Jersey,Public,Central,Not Ivy
Trinity,Private,North,Not Ivy
Tufts,Private,North,Not Ivy
Tulane,Private,South,Not Ivy
Tulsa,Private,Central,Not Ivy
UCD L&H,International,Central,Not Ivy
UCLA,Public,Central,Not Ivy
UConn,Public,North,Not Ivy
UMBC,Public,South,Not Ivy
University of Alaska Anchorage,Public,Central,Not Ivy
University of Albany,Public,North,Not Ivy
University of British Columbia,International,Central,Not Ivy
University of Calgary,International,Central,Not Ivy
University of Chicago,Private,Central,Not Ivy
University of Delaware,Public,South,Not Ivy
University of Denver,Private,Central,Not Ivy
University of Guelph,International,Central,Not Ivy
University of Hawaii at Manoa,Public,Central,Not Ivy
University of Massachusetts,Public,North,Not Ivy
University of Michigan,Public,Central,Not Ivy
University of Minnesota,Public,Central,Not Ivy
University of New South Wales,International,Central,Not Ivy
University of North Carolina,Public,South,Not Ivy
University of Pittsburgh,Public,Central,Not Ivy
University of Southern California,Private,Central,Not Ivy
University of Sydney,International,Central,Not Ivy
University of the People,Private,Central,Not Ivy
University of Vermont,Public,North,Not Ivy
University of Virginia,Public,South,Not Ivy
University of Waterloo,International,Central,Not Ivy
UT Austin,Public,Central,Not Ivy
Vassar,Private,North,Not Ivy
Villanova,Private,Central,Not Ivy
Washington University in St. Louis,Private,Central,Not Ivy
Wellesley,Private,North,Not Ivy
Wesleyan,Private,North,Not Ivy
Western,International,Central,Not Ivy
Wilfred Laurier University,International,Central,Not Ivy
William and Mary,Public,South,Not Ivy
Williams,Private,North,Not Ivy
WSCC,Unclear,Central,Not Ivy
York University,International,Central,Not Ivy
Yale,Private,North,Ivy

Overwriting Data_Curation_Data/Schools.csv


The following reads the recorded tournaments from the APDA website.

In [22]:
from pathlib import Path
from bs4 import BeautifulSoup
from io import StringIO
import pandas as pd
import numpy as np
def get_pages():
    """
    Creates file paths based on the contents of the data folder. 
    """
    paths = []
    for i in range (1, 42):
        for html in Path("Tournaments").glob(f"APDA_Results_{i}.html"):
            paths.append(html)
    return paths
pages = get_pages()

In [20]:
# Creates a df with all tournaments since 2004
all_tournaments = []
for page in pages:
    with open(page, encoding="utf-8") as fp: 
        soup = BeautifulSoup(fp, "html.parser")
        
        target_table = None

        for table in soup.find_all("table"):
            headers = [th.get_text(strip=True) for th in table.find_all("th")]
            if headers == ["ID", "Name", "Date", "Season", "Teams", "Novice Debaters"]:
                target_table = table
                break

        if target_table is None:
            continue

        df = pd.read_html(StringIO(str(target_table)))[0]
        all_tournaments.append(df)

tournaments = pd.concat(all_tournaments, ignore_index=True)
tournaments = tournaments.drop(columns=["ID"])

bp_patterns = ["NAUDC", "WUDC", "USUDC", "NorthAms", "Worlds", "Northams"]
for bp in bp_patterns:
    tournaments = tournaments[~tournaments['Name'].str.contains(bp, na=False)]
    
special_patterns = [
    ("ProAms", "ProAms"),
    ("BIPOC", "BIPOC"),
    ("Expansion", "Expansion"),
    ("Gender Minority", "Gender Minority"),
    ("Novice", "Novice"),
    ("Nationals", "Nationals"),
    ("Nov", "Novice"),
]

tournaments["Special"] = ""

for pattern, replacement in special_patterns:
    mask = tournaments["Name"].str.contains(pattern, regex=True, na=False)
    tournaments.loc[mask, "Special"] = replacement

mask = tournaments["Name"].str.contains("Online", regex=True, na=False) 
tournaments.loc[mask, "Online"] = "Online"

cleaning_patterns = [" IV", " II", " I", "\(Online Proams\)", "ProAms", "\(Proams\)", "Proams", "\(BIPOC\)", "BIPOC", "Novice", "\(Gender Minority\)", "\(Expansion\)", "\(Online\)",
                     "\(Elections\)", "\(APDA Meeting\)", "Nationals", "Online", "Nov"]
tournaments["Name"] = tournaments["Name"].replace(cleaning_patterns, "", regex=True)
replace_patterns = [("Hopkins", "Johns Hopkins"),
                    ("Johns Johns Hopkins", "Johns Hopkins"),
                    ("UMD", "Maryland"),
                    ("GW", "George Washington"),
                    ("Swat", "Swarthmore"),
                    ("TCNJ", "The College of New Jersey"),
                    ("VirginiaI", "Virginia"),
                    ("BrownI", "Brown"),
                    ("/", " + "),
                    ("UVA", "University of Virginia"),
                    ("University of Virginia Hybrid", "University of Virginia"),
                    ("BU", "Boston University"),
                    ("AU", "American"),
                    ("GU", "Georgetown"),
                    ("NU", "Northeastern"),
                    ("NU", "Northeastern"),
                    ("BC", "Boston College"),
                    ("Chicago", "University of Chicago"),
                    ("University of University of Chicago", "University of Chicago"),
                    ("Binghamton", "Binghamton University"),
                    ("Binghamton University University", "Binghamton University"),
                    ("Pitt+CMU", "Pitt + CMU"),
                    ("Moody Biblenstitute", "Moody Bible Institute"),
                    ("Moody Bible", "Moody Bible Institute"),
                    ("Moody Bible Institute Institute", "Moody Bible Institute"),
                    ("W&M", "William and Mary"),
                    ("Providence College \(Brown\)", "Brown"),
                    ("UMass", "University of Massachusetts"),
                    ("UMBoston College", "UMBC"),
                    ("WashU", "Washington University"),
                    ("F&M", "Franklin and Marshall")]
for old, new in replace_patterns:
    tournaments["Name"] = tournaments["Name"].replace(old, new, regex=True)
tournaments["Name"] = tournaments["Name"].replace("  \+  ", " + ", regex=True)

tournaments["Name"] = tournaments["Name"].str.strip()
tournaments = tournaments.rename(columns={"Name":"name", "Date":"date","Season":"season","Teams":"teams", "Novice Debaters":"novice_debaters", "Special":"special","Online":"online"})
tournaments["teams"] = np.where(tournaments["teams"] == -1, np.nan, tournaments["teams"])
tournaments["novice_debaters"] = np.where(tournaments["novice_debaters"] == -1, np.nan, tournaments["novice_debaters"])
tournaments.insert(0, "id", range(1, len(tournaments) + 1))
tournaments.to_csv("Data_Curation_Data/Tournaments.csv", index=False)

I will now check for missing data.

In [21]:
tournaments.isna().sum()

id                   0
name                 0
date                 0
season               0
teams                4
novice_debaters      4
special              0
online             945
dtype: int64

There are 4 missing counts of teams and 4 missing counts of novice_debaters. I will try to impute the teams and novice counts to the average of the host school, however, first we will need to confirm that there is enough data to safely do so.

In [22]:
missing_by_school = (tournaments[tournaments["teams"].isna()]["name"].value_counts())
total_by_school = tournaments["name"].value_counts()

missing_rate_by_school = (missing_by_school / total_by_school * 100).dropna().sort_values(ascending=False)
print(missing_rate_by_school.head(15))

name
Yale    10.000000
Penn     4.545455
Name: count, dtype: float64


We can see a small amount of data is missing, this is promising, lets confirm with how many points of data we have for each tournament host.

In [23]:
for school in ["Yale", "Penn"]:
    n_available = tournaments[tournaments["name"] == school]["teams"].notna().sum()
    print(school, n_available, "Rows with rounds data available")

Yale 27 Rows with rounds data available
Penn 21 Rows with rounds data available


With over 20 rounds of data, we can safely impute the missing teams.

In [28]:
schools_to_impute = ["Yale", "Penn"]

mask = tournaments["name"].isin(schools_to_impute)
tournaments.loc[mask, "teams"] = (
    tournaments.loc[mask]
    .groupby("name")["teams"]
    .transform(lambda x: x.fillna(x.median()))
    .round()
)

tournament_rounds_clean = tournaments.dropna(subset=["teams"]).copy()
print(f"{tournaments['teams'].isna().sum()} still missing")

0 still missing


We will do the same now with novice debaters.

In [29]:
missing_by_school = (tournaments[tournaments["novice_debaters"].isna()]["name"].value_counts())
total_by_school = tournaments["name"].value_counts()

missing_rate_by_school = (missing_by_school / total_by_school * 100).dropna().sort_values(ascending=False)
print(missing_rate_by_school.head(15))

name
Yale    10.000000
Penn     4.545455
Name: count, dtype: float64


Since this is the same, we can proceed.

In [30]:
schools_to_impute = ["Yale", "Penn"]

mask = tournaments["name"].isin(schools_to_impute)
tournaments.loc[mask, "novice_debaters"] = (
    tournaments.loc[mask]
    .groupby("name")["novice_debaters"]
    .transform(lambda x: x.fillna(x.median()))
    .round()
)

tournament_rounds_clean = tournaments.dropna(subset=["novice_debaters"]).copy()
print(f"{tournaments['novice_debaters'].isna().sum()} still missing")
tournaments.to_csv("Data_Curation_Data/Tournaments.csv", index=False)

0 still missing


That concludes the end of the data collection and curation I have completed for now.